# Nhận dạng rò rỉ nước trên ống nhánh PVC áp lực thấp bằng đặc trưng âm thanh thủ công và học máy

**Handcrafted Acoustic Features and Machine Learning for Leak Detection on Low-Pressure PVC Service Pipes:
Frequency Band, Pre-processing and Minimal Feature Subset**

---

### Bối cảnh dữ liệu

Âm thanh thu bằng **gậy nghe + micro độ nhạy cao** trên **ống nhánh PVC Φ27** (từ ống phân phối tới đồng hồ
khách hàng), **áp suất 0,5–2 bar**; hai thư mục `leak/` và `noleak/`; mỗi mã `Rxxx` là một lần thu độc lập
(có thể gồm nhiều tệp).

### Câu hỏi nghiên cứu / Research questions

| | Câu hỏi | Thí nghiệm | Mục |
|---|---|---|---|
| **RQ1** | Dải tần nào chứa tín hiệu rò rỉ? | E1: LTAS, AUC từng dải 1/6 & 1/3 octave, quét dải bằng mô hình | 5 |
| **RQ2** | Tín hiệu đã xử lý (lọc, loại bất thường, loại âm không liên quan, khử nhiễu) hay tín hiệu thô cho nhận dạng tốt hơn? | E2: 7 điều kiện tín hiệu × 4 mô hình, ghép cặp; E4c khẳng định trên TEST | 7, 9.4 |
| **RQ3** | Tập đặc trưng nhỏ nhất nào giữ được hiệu năng? | E3: xếp hạng đồng thuận trong fold, đường cong AUC(k), độ ổn định | 8 |
| **RQ4** | Mô hình có bền, diễn giải được và dùng được ngoài hiện trường không? | E4–E5: CV lồng, TEST khoá, SHAP, confound mức, rò rỉ dữ liệu, thời gian nghe tối thiểu | 9–10 |

### Quy trình / Pipeline

```
WAV ─► QC tệp ─► C1 band-pass + notch ─► C2 loại khung bất thường ─► C3 loại âm không liên quan ─► C4/C5/C6 khử nhiễu
                                         (va chạm, xung, bão hoà)      (tiếng nói, xe, sự kiện)       (so sánh 3 cách)
  ─► khung 2 s ─► ~170 đặc trưng thủ công ─► [trong từng fold] chuẩn hoá → xếp hạng đồng thuận → top-k → SVM | RF | XGBoost | k-NN
  ─► gộp xác suất về mức BẢN GHI ─► CV lặp nhóm theo bản ghi (DEV) + tập KIỂM TRA KHOÁ (TEST)
```

### Cách chạy trên Kaggle

1. `Add Data` → gắn dataset có hai thư mục `leak/` và `noleak/`; sửa `cfg.data_root` ở mục 1.2 (để sai thì
   notebook tự dò trong `/kaggle/input`; không thấy dữ liệu → **chế độ mô phỏng**, số liệu không có giá trị).
2. Không cần GPU. `Settings → Accelerator = None` (CPU 4 lõi) là đủ; bật Internet nếu thiếu `xgboost`/`PyWavelets`.
3. `Run All`. Cấu hình mặc định (7 điều kiện, 3×5-fold, CV lồng) mất khoảng **40–80 phút** trên CPU Kaggle.
   Đặc trưng được cache trong `leak_ml/cache/` nên chạy lại chỉ mất phần học máy.
4. Kết quả: `/kaggle/working/leak_ml/` (hình 300 dpi PNG+PDF, bảng CSV+LaTeX, `RESULTS_summary.md`,
   `key_results.json`, mô hình `.joblib`) và bản nén `leak_ml_outputs.zip`.

> Tài liệu phân tích phương pháp đi kèm: `docs/phan-tich-quy-trinh-ML-dac-trung-thu-cong.md` trong kho mã.

## 0. Cơ sở tài liệu / Literature basis

| Công trình | Đối tượng | Đặc trưng | Mô hình | Ghi chú dùng cho bài này |
|---|---|---|---|---|
| Hunaidi & Chu (1999), *Applied Acoustics* 58 | ống nhựa cấp nước | phổ tín hiệu rò rỉ | — | năng lượng rò rỉ ống nhựa tập trung ở tần số thấp, suy giảm mạnh theo khoảng cách |
| Gao et al. (2005), *J. Sound Vib.* 283 | ống nhựa chôn | — | — | ống nhựa hoạt động như bộ lọc thông thấp; lựa chọn cảm biến |
| Almeida et al. (2014), *Sensors* 14 | ống nhựa chôn | — | — | ảnh hưởng lọc của ống và cảm biến lên dải tần hữu ích |
| Martini et al. (2015), *Shock Vib.*; (2017), *Appl. Sci.* 7 | ống nhựa/PE đường kính nhỏ | đặc trưng rung/AE theo dải | ngưỡng/thống kê | gần nhất với ống nhánh Φ27 |
| Kang et al. (2018), *IEEE Trans. Ind. Electron.* 65 | mạng cấp nước | FFT/đặc trưng phổ | CNN + **SVM** | SVM là bộ phân loại chuẩn |
| El-Zahab et al. (2018), *MSSP* 108 | mạng cấp nước (gia tốc kế) | thống kê thời gian/tần số | NB, DT, **k-NN**, SVM… | tổ hợp đặc trưng thủ công + ML cổ điển |
| Cody, Dey & Narasimhan (2020), *J. Pipeline Syst. Eng. Pract.* 11 | ống cấp nước | **dự báo tuyến tính (LPC)** | phát hiện bất thường | cơ sở cho nhóm đặc trưng LPC |
| Ning et al. (2021), *Applied Acoustics* 182 | ống khí | MFCC + đặc trưng phổ/thời gian | **Random Forest** | quy trình trích + chọn đặc trưng + RF |
| Xu et al. (2021), *Process Saf. Environ. Prot.* 153 | ống | VMD + năng lượng/entropy | **SVM** | đặc trưng phân rã đa phân giải |
| Fan, Tariq & Zayed (2022), *Automation in Construction* 138 | tổng quan | — | — | SVM, ANN, RF, k-NN là các mô hình ML dùng nhiều nhất |
| Tijani et al. (2022), *Sci. Total Environ.* 821; Fares et al. (2023), *Environ. Technol.* 44 | mạng thực (noise logger/AE) | đặc trưng thời gian + tần số | SVM, RF, **boosting**, k-NN… | so sánh nhiều bộ phân loại, chọn đặc trưng |
| Cody, Tolson & Orchard (2020), *J. Comput. Civ. Eng.* 34 | mạng thực | phổ đồ thuỷ âm | autoencoder | mốc so sánh học sâu |

Khoảng trống: rất ít công trình về **ống nhánh PVC đường kính nhỏ, áp lực 0,5–2 bar, thu bằng gậy nghe**;
phần lớn chia train/test ngẫu nhiên theo đoạn (rò rỉ dữ liệu) và không kiểm định thống kê ở mức bản ghi;
hầu như không công trình nào **đo trực tiếp** tác động của từng bước tiền xử lý hay tìm **dải tần** và **tập
đặc trưng tối thiểu** bằng giao thức không thiên lệch. Danh mục tài liệu đầy đủ: xem tài liệu phân tích đi kèm.

## 1. Thiết lập môi trường / Environment setup

In [ ]:
# =========================================================
# 1.1 Thư viện / Imports
#     Chỉ cần CPU. Mọi thư viện đều có sẵn trên Kaggle; thiếu thì cài tự động.
# =========================================================
import os, sys, re, json, math, time, hashlib, warnings, subprocess, unicodedata, itertools, platform, shutil
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict

warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")


def _pip(pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return r.returncode == 0


try:
    import soundfile as sf
except Exception:
    _pip(["soundfile"]); import soundfile as sf
try:
    import pywt
except Exception:
    _pip(["PyWavelets"]); import pywt

import numpy as np
import pandas as pd
import scipy
import scipy.signal as sps
import scipy.stats as sst
from scipy.fft import dct, rfft, irfft
from scipy.special import expit
from scipy.cluster import hierarchy as sch
from scipy.spatial.distance import squareform
import librosa
import librosa.display
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Markdown
from joblib import Parallel, delayed
import joblib

import sklearn
from sklearn.model_selection import StratifiedKFold, KFold, ParameterGrid, train_test_split
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score, precision_recall_curve,
                             confusion_matrix, matthews_corrcoef)

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = _pip(["xgboost"])
    if HAS_XGB:
        import xgboost as xgb
try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

N_CPU = os.cpu_count() or 2
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | SciPy {scipy.__version__} | "
      f"pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"librosa {librosa.__version__} | PyWavelets {pywt.__version__} | "
      f"xgboost {xgb.__version__ if HAS_XGB else 'KHÔNG CÓ -> dùng HistGradientBoosting'} | "
      f"shap {shap.__version__ if HAS_SHAP else 'không có -> dùng permutation importance'}")
print(f"CPU: {N_CPU} lõi | {platform.platform()}")

In [ ]:
# =========================================================
# 1.2 Cấu hình trung tâm / Central configuration
#     Mọi tham số thí nghiệm nằm ở đây. Các giá trị mặc định được CỐ ĐỊNH TRƯỚC
#     (pre-registered) - không chỉnh theo kết quả trên tập kiểm tra.
# =========================================================
@dataclass
class CFG:
    # ---- Dữ liệu ----------------------------------------------------------
    data_root: str = "/kaggle/input/datasets/nguyennguyenkhac/tapdulieutt-8k-v7-2/field_rec_8k_V7.2"
    leak_keys: tuple = ("leak", "ro_ri", "rori")
    noleak_keys: tuple = ("noleak", "no_leak", "non_leak", "nonleak", "normal", "background",
                          "khong_ro_ri", "khongrori")
    audio_ext: tuple = (".wav", ".flac", ".ogg", ".mp3")
    record_regex: str = r"(R\d{1,4})"      # mã bản ghi = ĐƠN VỊ ĐỘC LẬP (nhóm của CV)

    # ---- Tín hiệu & khung phân tích ---------------------------------------
    target_sr: int = 16000                 # dữ liệu gốc 16 kHz -> giữ nguyên để xét tới 8 kHz
    frame_sec: float = 2.0                 # độ dài một khung phân loại
    hop_sec: float = 2.0                   # không chồng lấn: khung gần như độc lập về thời gian
    f_lo: float = 20.0                     # dải phân tích đặc trưng [f_lo, f_hi]
    f_hi: float = 7600.0                   # tự kẹp <= 0,95 x Nyquist

    # ---- C1: lọc điều kiện hoá (band-pass + notch lưới điện) --------------
    hp_hz: float = 20.0
    lp_hz: float = 7000.0
    filt_order: int = 4
    notch_hz: float = 50.0                 # lưới điện Việt Nam
    notch_harmonics: int = 6
    notch_q: float = 30.0

    # ---- QC mức tệp (bỏ cả tệp) ------------------------------------------
    clip_level: float = 0.999              # |x| >= mức này (full-scale = 1) coi là bão hoà
    qc_max_clip_frac: float = 0.01         # > 1% mẫu bão hoà -> loại tệp
    qc_max_zero_frac: float = 0.5          # > 50% mẫu bằng 0 (mất tín hiệu) -> loại tệp

    # ---- C2: loại khung bất thường (va chạm, xung, bão hoà, mất tín hiệu) --
    spike_k: float = 8.0                   # kẹp mẫu vượt k x sigma_MAD của tệp
    block_ms: float = 50.0                 # khối con để dò sự kiện năng lượng
    z_thr: float = 3.5                     # modified z-score (Iglewicz & Hoaglin, 1993)
    event_db: float = 6.0                  # VÀ vượt trung vị tệp >= 6 dB mới tính là sự kiện
    kurt_thr: float = 8.0                  # kurtosis (Gauss = 3) vượt ngưỡng -> khung xung
    frame_clip_frac: float = 0.001
    # ---- C3: loại âm thanh không liên quan (tiếng nói, xe, chim...) --------
    lsd_z_thr: float = 3.5                 # khoảng cách log-phổ tới phổ trung vị của tệp
    lsd_min_db: float = 3.0
    voicing_thr: float = 0.5               # đỉnh ACF chuẩn hoá (70-500 Hz) -> khối "hữu thanh"
    voicing_frac_thr: float = 0.3
    voicing_excess: float = 0.2            # phải cao hơn mức nền của chính tệp đó
    min_keep_frac: float = 0.3             # luôn giữ >= 30% số khung của mỗi tệp
    min_keep_frames: int = 3

    # ---- Khử nhiễu (C4-C6) -------------------------------------------------
    stft_nfft: int = 1024
    stft_hop: int = 256
    mbg_beta: float = 2.0                  # C4: kẹp biên độ > beta x trung vị theo thời gian
    ss_alpha: float = 2.0                  # C5: trừ phổ (Boll 1979; Berouti 1979)
    ss_floor: float = 0.1
    ss_quantile: float = 0.2
    wt_wavelet: str = "db8"                # C6: VisuShrink soft-threshold (Donoho & Johnstone 1994)
    wt_level: int = 6
    conditions: tuple = ("C0_RAW", "C1_BPF", "C2_TR", "C3_IR", "C4_MBG", "C5_SS", "C6_WT")

    # ---- Chia dữ liệu & CV ------------------------------------------------
    holdout_frac: float = 0.2              # tập KIỂM TRA KHOÁ (mức bản ghi); 0 = không dùng
    split_seed: int = 2024
    n_splits: int = 5
    seeds: tuple = (42, 43, 44)            # CV lặp 3 lần x 5 fold
    record_balanced: bool = True           # mỗi bản ghi đóng góp như nhau, mỗi lớp như nhau

    # ---- Mô hình -----------------------------------------------------------
    models: tuple = ("SVM", "RF", "XGB", "KNN")   # thêm "LR", "MLP" nếu muốn

    # ---- E1: dải tần -------------------------------------------------------
    e1_scope: str = "all"                  # "all" (mô tả, không dẫn tới quyết định nào) | "dev"
    e1_condition: str = "C3_IR"            # tín hiệu sạch nhiễu xung/không liên quan, CHƯA khử nhiễu
    e1_seeds: tuple = (42, 43)
    e1_tol: float = 0.02                   # sai khác AUC chấp nhận khi tìm "dải thiết yếu"

    # ---- E2: xử lý vs thô --------------------------------------------------
    e2_regimes: tuple = ("all", "mrmr30")  # toàn bộ đặc trưng | 30 đặc trưng mRMR chọn TRONG fold
    e2_primary: str = "mrmr30"

    # ---- E3: chọn đặc trưng ------------------------------------------------
    k_grid: tuple = (3, 5, 8, 10, 15, 20, 25, 30, 40, 50, 75, 100, 0)   # 0 = tất cả
    k_main_range: tuple = (20, 50)
    k_tol: float = 0.01                    # "không ảnh hưởng nhiều" = AUC giảm <= 0,01
    rankers: tuple = ("anova", "mi", "mrmr", "et", "l1")
    method_cmp_k: tuple = (10, 20, 30)
    method_cmp_models: tuple = ("SVM", "RF")
    l1_C: float = 0.05
    rank_max_rows: int = 6000

    # ---- E4: mô hình cuối --------------------------------------------------
    inner_splits: int = 3
    run_compact: bool = True

    # ---- Thống kê & đầu ra -------------------------------------------------
    n_boot: int = 2000
    alpha: float = 0.05
    out_dir: str = "/kaggle/working/leak_ml"
    fig_dpi: int = 300
    n_jobs: int = 0                        # 0 = mọi lõi CPU
    smoke_test: bool = False


cfg = CFG()
if os.environ.get("LEAK_SMOKE") == "1":
    cfg.smoke_test = True
if os.environ.get("LEAK_DATA") is not None:
    cfg.data_root = os.environ["LEAK_DATA"]
if os.environ.get("LEAK_OUT"):
    cfg.out_dir = os.environ["LEAK_OUT"]
elif not Path("/kaggle").exists():
    cfg.out_dir = str(Path("./leak_ml_outputs").resolve())

# Lưới siêu tham số (E4, chọn bằng CV lồng theo bản ghi) và giá trị mặc định (E1-E3)
PARAM_GRID = {
    "SVM": {"C": [0.3, 1.0, 3.0, 10.0, 30.0], "gamma_mult": [0.3, 1.0, 3.0]},
    "RF":  {"max_features": ["sqrt", 0.3], "min_samples_leaf": [1, 5, 20]},
    "XGB": {"max_depth": [3, 5], "learning_rate": [0.05, 0.1], "min_child_weight": [1, 5]},
    "KNN": {"n_neighbors": [11, 21, 41, 81, 161], "weights": ["uniform", "distance"]},
    "LR":  {"C": [0.01, 0.1, 1.0, 10.0]},
    "MLP": {"alpha": [1e-4, 1e-3, 1e-2], "hidden_layer_sizes": [(64,), (64, 32)]},
}
DEFAULT_PARAMS = {
    "SVM": {"C": 1.0, "gamma_mult": 1.0},
    "RF":  {"n_estimators": 300, "max_features": "sqrt", "min_samples_leaf": 3},
    "XGB": {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.05, "subsample": 0.8,
            "colsample_bytree": 0.8, "min_child_weight": 1, "reg_lambda": 1.0},
    "KNN": {"n_neighbors": 41, "weights": "uniform"},
    "LR":  {"C": 0.1},
    "MLP": {"hidden_layer_sizes": (64, 32), "alpha": 1e-3},
}

if cfg.smoke_test:
    print("!! SMOKE TEST: CV 1x3, lưới k rút gọn, bootstrap 200 - CHỈ để kiểm tra pipeline.")
    cfg.seeds, cfg.n_splits, cfg.e1_seeds = (42,), 3, (42,)
    cfg.k_grid = (3, 5, 10, 20, 30, 0)
    cfg.method_cmp_k = (5, 10)
    cfg.n_boot = 200
    cfg.inner_splits = 2
    cfg.fig_dpi = 110
    PARAM_GRID.update({"SVM": {"C": [1.0, 10.0], "gamma_mult": [1.0]},
                       "RF": {"max_features": ["sqrt"], "min_samples_leaf": [3]},
                       "XGB": {"max_depth": [3], "learning_rate": [0.1], "min_child_weight": [1]},
                       "KNN": {"n_neighbors": [11, 41], "weights": ["uniform"]}})
    DEFAULT_PARAMS["RF"]["n_estimators"] = 150
    DEFAULT_PARAMS["XGB"]["n_estimators"] = 150

N_JOBS = N_CPU if cfg.n_jobs in (0, None) else int(cfg.n_jobs)
OUT = Path(cfg.out_dir)
FIG_DIR, TAB_DIR, CACHE_DIR, MODEL_DIR = (OUT / d for d in ("figures", "tables", "cache", "models"))
for d in (FIG_DIR, TAB_DIR, CACHE_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cfg).items()},
                 indent=1, ensure_ascii=False))
print(f"\nThư mục đầu ra: {OUT} | song song: {N_JOBS} tiến trình")

In [ ]:
# =========================================================
# 1.3 Phong cách hình vẽ, lưu hình/bảng, tiện ích thống kê
# =========================================================
# Bảng màu phân loại (thứ tự CỐ ĐỊNH; màu đi theo thực thể, không theo thứ hạng)
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
PAL = {"noleak": CAT[0], "leak": CAT[1]}
MODEL_COLOR = {m: CAT[i] for i, m in enumerate(["SVM", "RF", "XGB", "KNN", "LR", "MLP"])}
COND_COLOR = {c: CAT[i] for i, c in enumerate(cfg.conditions)}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#f4f8fd", "#cde2fb", "#86b6ef", "#2a78d6", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div_br", ["#184f95", "#6da7ec", "#f0efec", "#ef8a8a", "#b52f2f"])

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": cfg.fig_dpi, "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.edgecolor": "#b9b8b3", "axes.linewidth": 0.6, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.5, "grid.linestyle": "-", "axes.axisbelow": True, "axes.spines.top": False,
    "axes.spines.right": False, "lines.linewidth": 1.6, "text.color": INK, "axes.labelcolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "legend.frameon": False, "figure.facecolor": "white",
})

FIG_LOG = []


def savefig(fig, name, caption=""):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=cfg.fig_dpi, bbox_inches="tight")
    FIG_LOG.append((name, caption))
    plt.show()
    plt.close(fig)


def freq_axis(ax, lo=None, hi=None, label="frequency (Hz)"):
    """Trục tần số log với nhãn dễ đọc (31.5 ... 8k)."""
    ax.set_xscale("log")
    ticks = [31.5, 63, 125, 250, 500, 1000, 2000, 4000, 8000]
    lo = lo or ax.get_xlim()[0]; hi = hi or ax.get_xlim()[1]
    tk = [t for t in ticks if lo <= t <= hi]
    ax.set_xticks(tk); ax.set_xticklabels([f"{t / 1000:g}k" if t >= 1000 else f"{t:g}" for t in tk])
    ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax.set_xlim(lo, hi)
    if label:
        ax.set_xlabel(label)


def savetab(df, name, index=False, floatfmt=4):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index)
    try:
        (TAB_DIR / f"{name}.tex").write_text(
            df.to_latex(index=index, float_format=lambda v: f"{v:.{floatfmt}f}", escape=True))
    except Exception:
        pass
    return df


def holm(p):
    """Hiệu chỉnh Holm-Bonferroni (Holm, 1979)."""
    p = np.asarray(p, float); m = len(p); out = np.full(m, np.nan)
    ok = np.isfinite(p); idx = np.where(ok)[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]; run = 0.0; mm = len(order)
    for i, j in enumerate(order):
        run = max(run, (mm - i) * p[j]); out[j] = min(1.0, run)
    return out


def bh_fdr(p):
    """q-value Benjamini-Hochberg (1995)."""
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    ranked = p[order] * m / np.arange(1, m + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(m); out[order] = np.minimum(q, 1.0)
    return out


def auc_safe(y, s):
    y = np.asarray(y); s = np.asarray(s, float)
    m = np.isfinite(s)
    if m.sum() < 2 or len(np.unique(y[m])) < 2:
        return np.nan
    return float(roc_auc_score(y[m], s[m]))


def cliffs_delta(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan
    u = sst.mannwhitneyu(a, b, alternative="two-sided").statistic
    return 2.0 * u / (len(a) * len(b)) - 1.0


def mwu_p(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return float(sst.mannwhitneyu(a, b, alternative="two-sided").pvalue)


def boot_auc_ci(y, s, n_boot=None, seed=0, groups=None, alpha=None):
    """CI bootstrap phân tầng của AUC. groups != None -> bootstrap THEO CỤM (bản ghi)."""
    n_boot = n_boot or cfg.n_boot; alpha = alpha or cfg.alpha
    y = np.asarray(y); s = np.asarray(s, float); rng = np.random.default_rng(seed)
    vals = []
    if groups is None:
        pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
        if len(pos) == 0 or len(neg) == 0:
            return np.nan, np.nan
        for _ in range(n_boot):
            i = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
            vals.append(roc_auc_score(y[i], s[i]))
    else:
        groups = np.asarray(groups)
        gids, inv = np.unique(groups, return_inverse=True)
        members = [np.where(inv == k)[0] for k in range(len(gids))]
        glab = np.array([y[m[0]] for m in members])
        gp, gn = np.where(glab == 1)[0], np.where(glab == 0)[0]
        if len(gp) == 0 or len(gn) == 0:
            return np.nan, np.nan
        for _ in range(n_boot):
            pick = np.concatenate([rng.choice(gp, len(gp)), rng.choice(gn, len(gn))])
            i = np.concatenate([members[k] for k in pick])
            vals.append(roc_auc_score(y[i], s[i]))
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def paired_boot_delta(y, s1, s2, n_boot=None, seed=0):
    """Delta AUC ghép cặp (cùng các bản ghi): CI bootstrap phân tầng + p hai phía."""
    n_boot = n_boot or cfg.n_boot
    y = np.asarray(y); s1 = np.asarray(s1, float); s2 = np.asarray(s2, float)
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    d = []
    for _ in range(n_boot):
        i = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        d.append(roc_auc_score(y[i], s1[i]) - roc_auc_score(y[i], s2[i]))
    d = np.asarray(d)
    lo, hi = np.percentile(d, [100 * cfg.alpha / 2, 100 * (1 - cfg.alpha / 2)])
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return float(lo), float(hi), float(min(1.0, max(p, 1.0 / n_boot)))


def _midrank(x):
    j = np.argsort(x, kind="mergesort"); z = x[j]; n = len(x); t = np.zeros(n); i = 0
    while i < n:
        k = i
        while k < n and z[k] == z[i]:
            k += 1
        t[i:k] = 0.5 * (i + k - 1) + 1
        i = k
    out = np.empty(n); out[j] = t
    return out


def delong_cov(y, scores):
    """AUC và hiệp phương sai theo DeLong et al. (1988), cài đặt nhanh của Sun & Xu (2014)."""
    y = np.asarray(y).astype(int); pos = y == 1; m, n = int(pos.sum()), int((~pos).sum())
    k = len(scores); v10 = np.zeros((k, m)); v01 = np.zeros((k, n)); aucs = np.zeros(k)
    for r, s in enumerate(scores):
        s = np.asarray(s, float); X, Y = s[pos], s[~pos]
        tx, ty, tz = _midrank(X), _midrank(Y), _midrank(np.concatenate([X, Y]))
        aucs[r] = (tz[:m].sum() - m * (m + 1) / 2) / (m * n)
        v10[r] = (tz[:m] - tx) / n
        v01[r] = 1.0 - (tz[m:] - ty) / m
    S = np.atleast_2d(np.cov(v10)) / m + np.atleast_2d(np.cov(v01)) / n
    return aucs, S


def delong_test(y, s1, s2):
    aucs, S = delong_cov(y, [s1, s2])
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    d = aucs[0] - aucs[1]
    if not np.isfinite(var) or var <= 1e-12:
        return float(d), (1.0 if abs(d) < 1e-12 else 0.0)
    return float(d), float(2 * sst.norm.sf(abs(d) / np.sqrt(var)))


def nb_ttest(d, n_train, n_test):
    """t-test hiệu chỉnh cho CV lặp (Nadeau & Bengio, 2003)."""
    d = np.asarray(d, float); d = d[np.isfinite(d)]; J = len(d)
    if J < 2 or np.var(d, ddof=1) == 0:
        return np.nan
    t = d.mean() / np.sqrt((1.0 / J + n_test / n_train) * np.var(d, ddof=1))
    return float(2 * sst.t.sf(abs(t), J - 1))


def bin_metrics(y, p, thr=0.5):
    y = np.asarray(y).astype(int); yhat = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    prec = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan
    return dict(acc=(tp + tn) / len(y), bacc=np.nanmean([sens, spec]), sens=sens, spec=spec,
                prec=prec, f1=f1, mcc=matthews_corrcoef(y, yhat) if len(np.unique(yhat)) > 1 else 0.0,
                tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))


def fmt_ci(v, lo, hi, d=3):
    return f"{v:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]" if np.isfinite(v) else "n/a"


print("Tiện ích thống kê sẵn sàng: Holm, BH-FDR, bootstrap cụm, DeLong, Nadeau-Bengio.")

## 2. Dữ liệu: kiểm kê, định danh bản ghi, chia tập / Data inventory and splitting

- Nhãn lấy từ **tên thư mục** (`leak` → 1, `noleak` → 0).
- Mỗi mã `Rxxx` là **một lần thu độc lập**; một lần thu có thể bị cắt thành nhiều tệp
  (`R001_leak01.wav`, `R001_leak02.wav`...). Mọi khung của cùng một bản ghi luôn nằm cùng một
  phía train/test → **không rò rỉ dữ liệu** giữa các khung chồng/kề nhau.
- **Tập kiểm tra khoá (hold-out)**: `cfg.holdout_frac` số bản ghi (phân tầng theo lớp) được tách ra
  ngay từ đầu và **không** tham gia bất kỳ quyết định nào (chọn điều kiện xử lý, số đặc trưng,
  siêu tham số). Chỉ dùng đúng một lần ở mục 9.

In [ ]:
# =========================================================
# 2.1 Dò thư mục dữ liệu + chế độ dữ liệu mô phỏng (khi không có dữ liệu thật)
# =========================================================
def _norm_token(s):
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return s.lower().replace("-", "_").replace(" ", "_")


def _key_regex(keys):
    ks = sorted({_norm_token(k) for k in keys}, key=len, reverse=True)
    return re.compile(r"(?:^|[_.])(?:" + "|".join(map(re.escape, ks)) + r")(?=$|[_.\d])")


_NOLEAK_RE, _LEAK_RE = _key_regex(cfg.noleak_keys), _key_regex(cfg.leak_keys)


def label_of_token(s):
    s = _norm_token(s)
    if _NOLEAK_RE.search(s):          # xét 'noleak' TRƯỚC 'leak'
        return 0
    if _LEAK_RE.search(s):
        return 1
    return None


def label_from_path(p):
    for tok in [p.parent.name, p.stem] + list(reversed(p.parts[:-2])):
        lab = label_of_token(tok)
        if lab is not None:
            return lab
    return None


for _t, _w in [("leak", 1), ("noleak", 0), ("No-Leak", 0), ("R001_leak01", 1), ("R12_noleak_03", 0),
               ("rò_rỉ", 1), ("leakage", None), ("field_rec_8k_V7.2", None)]:
    assert label_of_token(_t) == _w, f"bộ gán nhãn sai với '{_t}'"


def audio_files_in(root):
    return sorted(p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in cfg.audio_ext)


def find_data_root(explicit):
    if explicit and Path(explicit).exists():
        fs = audio_files_in(explicit)
        if {0, 1}.issubset({label_from_path(p) for p in fs}):
            return Path(explicit)
        print(f"!! {explicit} tồn tại nhưng không chứa đủ hai lớp leak/noleak.")
    elif explicit:
        print(f"!! cfg.data_root = {explicit} KHÔNG tồn tại -> tự dò trong /kaggle/input.")
    cands = []
    for base in [Path("/kaggle/input"), Path("./data")]:
        if not base.exists():
            continue
        for d in [base] + [p for p in base.rglob("*") if p.is_dir()]:
            try:
                labs = {label_of_token(k.name) for k in d.iterdir() if k.is_dir()}
            except Exception:
                continue
            if {0, 1}.issubset(labs):
                cands.append(d)
    if not cands:
        return None
    cands.sort(key=lambda p: (-len(p.parts), str(p)))
    return cands[0]


def synthesize_dataset(root, n_per_class=14, sr=16000, seed=7):
    """Dữ liệu MÔ PHỎNG để kiểm tra pipeline (KHÔNG dùng để báo cáo khoa học).
    Rò rỉ = ồn băng rộng 250-2500 Hz (+ đôi khi tiếng rít); nền = ồn hồng + hài 50 Hz + tiếng bơm.
    Cả hai lớp đều chứa nhiễu xung (va chạm gậy), đoạn giống tiếng nói và đôi khi bão hoà."""
    rng = np.random.default_rng(seed)
    root = Path(root)
    if root.exists():
        shutil.rmtree(root, ignore_errors=True)
    rid = 0
    for cls in ("leak", "noleak"):
        (root / cls).mkdir(parents=True, exist_ok=True)
        for _ in range(n_per_class):
            rid += 1
            gain = 10 ** (rng.uniform(-6, 6) / 20)
            snr = rng.uniform(-26, -10)
            tilt = rng.uniform(-0.6, 0.6)                   # "dấu vân tay" vị trí thu: độ nghiêng phổ riêng
            flow = cls == "noleak" and rng.random() < 0.3   # ồn dòng chảy dải thấp gây nhầm
            f1, f2 = rng.uniform(250, 700), rng.uniform(1500, 2500)
            whistle = rng.random() < 0.3
            pump = rng.random() < 0.3
            for k in range(int(rng.integers(1, 4))):
                dur = rng.uniform(12, 24) if cfg.smoke_test else rng.uniform(20, 45)
                n = int(dur * sr); t = np.arange(n) / sr
                w = rng.standard_normal(n)
                W = np.fft.rfft(w); fr = np.fft.rfftfreq(n, 1 / sr); W[1:] /= np.sqrt(fr[1:]); W[0] = 0
                W[1:] *= (fr[1:] / 1000.0) ** tilt
                bg = np.fft.irfft(W, n); bg /= bg.std()
                bg += rng.uniform(0, 0.3) * sum(np.sin(2 * np.pi * 50 * h * t + rng.uniform(0, 6)) / h
                                                for h in range(1, 5))
                if pump:
                    f0 = rng.uniform(90, 160)
                    bg += 0.3 * sum(np.sin(2 * np.pi * f0 * h * t) / h for h in range(1, 6))
                x = bg.copy()
                if flow:
                    fl = sps.sosfilt(sps.butter(4, [100, 700], btype="bandpass", fs=sr, output="sos"), rng.standard_normal(n))
                    x += fl / fl.std() * 10 ** (rng.uniform(-20, -10) / 20)
                if cls == "leak":
                    sos = sps.butter(4, [f1, f2], btype="bandpass", fs=sr, output="sos")
                    lk = sps.sosfilt(sos, rng.standard_normal(n)); lk /= lk.std()
                    if whistle:
                        lk += 0.4 * np.sin(2 * np.pi * rng.uniform(900, 1400) * t + 0.3 * np.sin(2 * np.pi * 0.5 * t))
                    x += lk * 10 ** (snr / 20)
                for _k in range(rng.poisson(1.5)):          # va chạm gậy nghe
                    i0 = int(rng.integers(0, n - sr // 4)); L = sr // 4
                    x[i0:i0 + L] += rng.uniform(6, 20) * np.exp(-np.arange(L) / (0.03 * sr)) * \
                        np.sin(2 * np.pi * rng.uniform(100, 600) * np.arange(L) / sr)
                for _k in range(rng.poisson(0.7)):          # đoạn giống tiếng nói
                    L = int(rng.uniform(0.8, 2.0) * sr); i0 = int(rng.integers(0, max(1, n - L)))
                    tt = np.arange(L) / sr; f0 = rng.uniform(110, 220)
                    v = sum(np.sin(2 * np.pi * f0 * h * tt) * np.exp(-((f0 * h - 700) / 900) ** 2) for h in range(1, 12))
                    x[i0:i0 + L] += rng.uniform(1.5, 4) * v * (0.5 + 0.5 * np.sin(2 * np.pi * 4 * tt)) / (np.std(v) + 1e-9)
                x = 0.05 * gain * x / x.std()
                if rng.random() < 0.1:                      # bão hoà ngắn
                    i0 = int(rng.integers(0, n - sr // 2)); x[i0:i0 + sr // 4] *= 40
                sf.write(str(root / cls / f"R{rid:03d}_{cls}{k + 1:02d}.wav"), np.clip(x, -1, 1).astype(np.float32),
                         sr, subtype="PCM_16")
    return root


DATA_ROOT = find_data_root(cfg.data_root)
SYNTHETIC = DATA_ROOT is None
if SYNTHETIC:
    print("!! KHÔNG tìm thấy dữ liệu leak/noleak -> SYNTHETIC DEMO MODE (số liệu KHÔNG có giá trị khoa học).")
    DATA_ROOT = synthesize_dataset(OUT / "synthetic_data",
                                   n_per_class=int(os.environ.get("LEAK_SYNTH_N", 10 if cfg.smoke_test else 16)))
print("Thư mục dữ liệu:", DATA_ROOT)

In [ ]:
# =========================================================
# 2.2 Bảng kiểm kê mức TỆP và mức BẢN GHI
# =========================================================
_REC_RE = re.compile(cfg.record_regex, re.IGNORECASE)
rows = []
for p in audio_files_in(DATA_ROOT):
    lab = label_from_path(p)
    if lab is None:
        continue
    try:
        info = sf.info(str(p))
    except Exception as e:
        print("  [bỏ qua]", p.name, type(e).__name__); continue
    m = _REC_RE.search(p.stem) or _REC_RE.search(str(p.parent))
    rows.append(dict(path=str(p), file_stem=p.stem, record_id=(m.group(1).upper() if m else p.stem),
                     rid_from_regex=bool(m), label=lab, cls="leak" if lab == 1 else "noleak",
                     duration_s=info.duration, sr_native=info.samplerate, channels=info.channels))
files = pd.DataFrame(rows).sort_values(["cls", "record_id", "file_stem"]).reset_index(drop=True)
assert len(files) > 0, "Không đọc được tệp âm thanh nào"

conf = files.groupby("record_id").label.nunique()
if (conf > 1).any():
    raise ValueError(f"Mã bản ghi mang HAI nhãn: {conf[conf > 1].index.tolist()} - sửa dữ liệu hoặc cfg.record_regex")
if (~files.rid_from_regex).any():
    print(f"!! {int((~files.rid_from_regex).sum())} tệp không khớp regex {cfg.record_regex} -> dùng tên tệp làm mã bản ghi.")

records = (files.groupby("record_id")
           .agg(cls=("cls", "first"), label=("label", "first"), n_files=("file_stem", "size"),
                duration_s=("duration_s", "sum"), sr=("sr_native", "min")).reset_index())

# ---- Tách tập KIỂM TRA KHOÁ ở mức bản ghi (phân tầng theo lớp) -----------------
if cfg.holdout_frac and cfg.holdout_frac > 0:
    _dev, _test = train_test_split(records.record_id.values, test_size=cfg.holdout_frac,
                                   stratify=records.label.values, random_state=cfg.split_seed)
    records["split"] = np.where(records.record_id.isin(_test), "test", "dev")
else:
    records["split"] = "dev"
files = files.merge(records[["record_id", "split"]], on="record_id")
HAS_TEST = (records.split == "test").any()

tab1 = (records.groupby(["split", "cls"])
        .agg(records=("record_id", "nunique"), files=("n_files", "sum"),
             total_min=("duration_s", lambda s: s.sum() / 60), mean_s=("duration_s", "mean"),
             min_s=("duration_s", "min"), max_s=("duration_s", "max")).round(1).reset_index())
display(Markdown("**Bảng 1. Kiểm kê dữ liệu mức bản ghi (đơn vị thống kê độc lập).**"))
display(tab1)
savetab(tab1, "T01_inventory")
print(f"Tệp: {len(files)} | Bản ghi: {len(records)} "
      f"({(records.label == 1).sum()} leak / {(records.label == 0).sum()} noleak) | "
      f"tổng {records.duration_s.sum() / 60:.1f} phút | sr gốc: {sorted(files.sr_native.unique())} Hz")
print(f"DEV: {(records.split == 'dev').sum()} bản ghi | TEST (khoá): {(records.split == 'test').sum()} bản ghi")
if files.sr_native.max() < 2 * cfg.f_hi:
    print(f"!! Tần số lấy mẫu gốc {files.sr_native.max()} Hz < 2 x f_hi: dải phân tích sẽ bị kẹp theo Nyquist.")

In [ ]:
# =========================================================
# 2.3 Hình 1: Tổng quan dữ liệu
# =========================================================
fig, ax = plt.subplots(1, 3, figsize=(11, 3.0))
cnt = records.groupby(["cls", "split"]).size().unstack(fill_value=0).reindex(["noleak", "leak"])
bottom = np.zeros(2)
for j, sp in enumerate([s for s in ("dev", "test") if s in cnt.columns]):
    ax[0].bar(cnt.index, cnt[sp].values, bottom=bottom, width=0.55,
              color=[PAL[c] for c in cnt.index], alpha=1.0 if sp == "dev" else 0.45,
              edgecolor="white", linewidth=2, label=sp.upper())
    for i, v in enumerate(cnt[sp].values):
        if v:
            ax[0].text(i, bottom[i] + v / 2, str(v), ha="center", va="center", color="white", fontsize=8)
    bottom += cnt[sp].values
ax[0].set_title("(a) Recordings per class (DEV / TEST)"); ax[0].set_ylabel("recordings")
ax[0].legend(loc="upper left")
bins = np.linspace(0, records.duration_s.max() * 1.05, 16)
for c in ("noleak", "leak"):
    ax[1].hist(records.loc[records.cls == c, "duration_s"], bins=bins, color=PAL[c], alpha=0.65,
               label=c, edgecolor="white", linewidth=1)
ax[1].set_title("(b) Recording duration"); ax[1].set_xlabel("s"); ax[1].legend()
fr = records.groupby(["n_files", "cls"]).size().unstack(fill_value=0).reindex(columns=["noleak", "leak"], fill_value=0)
xx = np.arange(len(fr))
for j, c in enumerate(["noleak", "leak"]):
    ax[2].bar(xx + (j - 0.5) * 0.36, fr[c].values, width=0.34, color=PAL[c], label=c)
ax[2].set_xticks(xx); ax[2].set_xticklabels(fr.index); ax[2].set_xlabel("files per recording")
ax[2].set_title("(c) Files per recording"); ax[2].legend()
fig.tight_layout()
savefig(fig, "Fig01_dataset_overview", "Dataset overview at recording level.")

## 3. Chuỗi xử lý tín hiệu / Signal-processing chain

Bảy **điều kiện tín hiệu** được so sánh trong cùng một khung đánh giá (mục 7 — câu hỏi RQ2):

| Mã | Thành phần | Mục đích |
|---|---|---|
| `C0_RAW` | chỉ giải mã + (nếu cần) tái lấy mẫu | **tín hiệu thô**, không xử lý gì |
| `C1_BPF` | khử DC + band-pass Butterworth pha-không 20–7000 Hz + notch 50 Hz × 6 hài | loại trôi nền/hạ âm, nhiễu điện lưới, nhiễu gần Nyquist |
| `C2_TR` | C1 + kẹp xung mẫu (k·σ_MAD) + **loại khung bất thường**: bão hoà, mất tín hiệu, sự kiện năng lượng (va chạm gậy nghe), kurtosis cao | loại nhiễu xung / tín hiệu bất thường |
| `C3_IR` | C2 + **loại khung không liên quan**: phổ lệch khỏi phổ nền của chính tệp (LSD), khung hữu thanh (tiếng nói, chim, động cơ gián đoạn) | loại âm thanh không liên quan tới rò rỉ |
| `C4_MBG` | C3 + **khử nhiễu không dừng** bằng kẹp biên độ STFT về *trung vị theo thời gian* (giữ thành phần dừng) | khử nhiễu **bảo toàn** tiếng rò rỉ (đề xuất) |
| `C5_SS` | C3 + **trừ phổ dừng** (Boll 1979; Berouti 1979) với phổ nhiễu ước lượng từ chính tệp | khử nhiễu "kinh điển" kiểu tiếng nói |
| `C6_WT` | C3 + **khử nhiễu wavelet** soft-threshold VisuShrink (Donoho & Johnstone 1994) | khử nhiễu phổ biến trong các bài báo AE/rò rỉ |

**Nguyên tắc quan trọng**

1. Mọi quy tắc loại bỏ đều **không nhìn nhãn** (label-blind) và dùng **ngưỡng tương đối so với chính
   tệp đó** (trung vị/MAD) → áp dụng giống hệt cho leak và noleak, không tạo khác biệt giả giữa hai lớp.
2. Tiếng rò rỉ là **ồn băng rộng, dừng** (stationary). Các bộ khử nhiễu kiểu tiếng nói (C5, C6) coi mọi
   thành phần dừng là "nhiễu nền" → **có nguy cơ xoá chính tín hiệu rò rỉ**. C4 làm điều ngược lại: giữ
   phần dừng, chỉ cắt phần *nhô lên nhất thời* (tiếng nói, xe, va chạm). Thí nghiệm E2 kiểm chứng giả
   thuyết này bằng số liệu.
3. Mỗi tệp luôn giữ ≥ `min_keep_frac` số khung → không bản ghi nào biến mất khỏi đánh giá.

In [ ]:
# =========================================================
# 3.1 Các khối xử lý / Processing blocks
# =========================================================
from numpy.lib.stride_tricks import sliding_window_view


def load_raw(path, sr):
    x, sr0 = sf.read(str(path), always_2d=True, dtype="float32")
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).mean(axis=1).astype(np.float64)
    if sr0 != sr:
        g = math.gcd(int(sr0), int(sr))
        x = sps.resample_poly(x, sr // g, sr0 // g)
    return x


def interp_weights(f, edges):
    """Trọng số nội suy tuyến tính để lấy tích phân luỹ tích tại các biên dải (dùng chung cho nhiều khung)."""
    e = np.clip(np.asarray(edges, float), f[0], f[-1])
    i1 = np.clip(np.searchsorted(f, e), 1, len(f) - 1); i0 = i1 - 1
    w = (e - f[i0]) / (f[i1] - f[i0])
    return i0, i1, w


def band_powers(f, P, lo, hi, _cache={}):
    """Công suất trong các dải [lo, hi] từ PSD (tích phân hình thang + nội suy biên). P: (..., nfreq)."""
    key = (len(f), float(f[1] - f[0]), tuple(np.round(lo, 3)), tuple(np.round(hi, 3)))
    if key not in _cache:
        _cache[key] = (interp_weights(f, lo), interp_weights(f, hi))
    (a0, a1, aw), (b0, b1, bw) = _cache[key]
    df = f[1] - f[0]
    cum = np.concatenate([np.zeros(P.shape[:-1] + (1,)), np.cumsum(0.5 * (P[..., 1:] + P[..., :-1]) * df, axis=-1)], axis=-1)
    ca = cum[..., a0] * (1 - aw) + cum[..., a1] * aw
    cb = cum[..., b0] * (1 - bw) + cum[..., b1] * bw
    return np.maximum(cb - ca, 0.0)


THIRD_OCT_NOMINAL = [25, 31.5, 40, 50, 63, 80, 100, 125, 160, 200, 250, 315, 400, 500, 630, 800, 1000,
                     1250, 1600, 2000, 2500, 3150, 4000, 5000, 6300, 8000]


def third_octave_bands(f_lo, f_hi):
    c = 1000.0 * 2 ** (np.arange(-16, 10) / 3)
    lo, hi = c * 2 ** (-1 / 6), c * 2 ** (1 / 6)
    m = (lo >= f_lo * 0.9) & (hi <= f_hi)
    return lo[m], hi[m], [THIRD_OCT_NOMINAL[i] for i in np.where(m)[0]]


class SignalChain:
    """Các bước xử lý của một tệp. Không dùng nhãn, không dùng thống kê chéo giữa các tệp."""

    def __init__(self, sr, P):
        self.sr, self.P = sr, P
        nyq = sr / 2
        self.f_hi = min(P["f_hi"], 0.95 * nyq)
        lp = min(P["lp_hz"], 0.95 * nyq)
        self.sos_bp = sps.butter(P["filt_order"], [P["hp_hz"], lp], btype="bandpass", fs=sr, output="sos")
        self.notches = [sps.iirnotch(h * P["notch_hz"], P["notch_q"], fs=sr)
                        for h in range(1, P["notch_harmonics"] + 1) if h * P["notch_hz"] < 0.95 * nyq]
        self.sos_voice = sps.butter(4, [60, min(1000, 0.9 * nyq)], btype="bandpass", fs=sr, output="sos")
        self.F = int(round(P["frame_sec"] * sr)); self.H = int(round(P["hop_sec"] * sr))
        self.B = int(round(P["block_ms"] * sr / 1000))
        self.lsd_lo, self.lsd_hi, _ = third_octave_bands(P["f_lo"], self.f_hi)

    # ---- C1 -------------------------------------------------------------
    def condition(self, x):
        y = sps.sosfiltfilt(self.sos_bp, x - x.mean())
        for b, a in self.notches:
            y = sps.filtfilt(b, a, y)
        return y

    # ---- C2: kẹp xung ở mức mẫu --------------------------------------------
    def spike_clip(self, x):
        s = 1.4826 * np.median(np.abs(x - np.median(x))) + 1e-12
        k = self.P["spike_k"] * s
        return np.clip(x, -k, k), float((np.abs(x) > k).mean())

    def frame_starts(self, n):
        return np.arange(0, n - self.F + 1, self.H) if n >= self.F else np.array([], int)

    # ---- C2/C3: thống kê chất lượng từng khung + mặt nạ ----------------------
    def qc_frames(self, xr, x1, starts):
        P, F, B, sr = self.P, self.F, self.B, self.sr
        nf = len(starts)
        idx = starts[:, None] + np.arange(F)[None, :]
        fr_raw, fr1 = xr[idx], x1[idx]
        clip_frac = (np.abs(fr_raw) >= P["clip_level"]).mean(1)
        zero_frac = (fr_raw == 0).mean(1)
        rms_raw = np.sqrt((fr_raw ** 2).mean(1))
        kurt = sst.kurtosis(fr1, axis=1, fisher=False)
        # (a) sự kiện năng lượng: khối 50 ms mạnh nhất trong khung so với trung vị cả tệp
        nb = len(x1) // B
        blk = 10 * np.log10((x1[:nb * B].reshape(nb, B) ** 2).mean(1) + 1e-20)
        med_b = np.median(blk); mad_b = 1.4826 * np.median(np.abs(blk - med_b)) + 1e-9
        nbf = max(1, F // B)
        blkmax = np.array([blk[s // B: s // B + nbf].max() for s in starts])
        e_excess = blkmax - med_b; e_z = e_excess / mad_b
        # (b) khoảng cách log-phổ (1/3 octave) tới phổ trung vị của tệp
        f, Pw = sps.welch(fr1, fs=sr, nperseg=min(1024, F), axis=-1)
        Ldb = 10 * np.log10(band_powers(f, Pw, self.lsd_lo, self.lsd_hi) + 1e-20)
        lsd = np.sqrt(((Ldb - np.median(Ldb, 0)) ** 2).mean(1))
        lsd_z = (lsd - np.median(lsd)) / (1.4826 * np.median(np.abs(lsd - np.median(lsd))) + 1e-9)
        # (c) mức hữu thanh: đỉnh ACF chuẩn hoá (không chệch) ở trễ 2-14 ms, khối 40 ms bước 20 ms
        xv = sps.sosfiltfilt(self.sos_voice, x1)
        L, step = int(0.04 * sr), int(0.02 * sr)
        sub = sliding_window_view(xv, L)[::step]
        sub = sub - sub.mean(1, keepdims=True)
        nfft = 1 << int(np.ceil(np.log2(2 * L)))
        r = np.fft.irfft(np.abs(np.fft.rfft(sub, nfft, axis=1)) ** 2, nfft, axis=1)[:, :L]
        r = r / (r[:, :1] + 1e-20) * (L / (L - np.arange(L)))
        voiced = r[:, int(sr / 500): int(sr / 70)].max(1) > P["voicing_thr"]
        centers = np.arange(len(sub)) * step + L // 2
        vfrac = np.array([voiced[(centers >= s) & (centers < s + F)].mean()
                          if ((centers >= s) & (centers < s + F)).any() else 0.0 for s in starts])
        q = pd.DataFrame(dict(clip_frac=clip_frac, zero_frac=zero_frac, rms_raw=rms_raw, kurt=kurt,
                              e_excess_db=e_excess, e_z=e_z, lsd_db=lsd, lsd_z=lsd_z, voiced_frac=vfrac))
        q["f_clip"] = clip_frac > P["frame_clip_frac"]
        q["f_silence"] = (rms_raw < 1e-5) | (zero_frac > 0.2)
        q["f_event"] = (e_z > P["z_thr"]) & (e_excess > P["event_db"])
        q["f_impulsive"] = kurt > P["kurt_thr"]
        q["f_spectral"] = (lsd_z > P["lsd_z_thr"]) & (lsd > P["lsd_min_db"])
        q["f_voiced"] = (vfrac > P["voicing_frac_thr"]) & (vfrac - np.median(vfrac) > P["voicing_excess"])
        abn = (q.f_clip | q.f_silence | q.f_event | q.f_impulsive).values
        irr = (q.f_spectral | q.f_voiced).values
        score = (10.0 * (q.f_clip | q.f_silence).values + np.maximum(0, e_z) / P["z_thr"]
                 + np.maximum(0, kurt - 3) / max(P["kurt_thr"] - 3, 1e-6)
                 + np.maximum(0, lsd_z) / P["lsd_z_thr"] + vfrac)
        keep2 = self._guard(~abn, score)
        keep3 = self._guard(keep2 & ~irr, score, prefer=keep2)
        q["keep_C2"], q["keep_C3"] = keep2, keep3
        q["guard_C2"] = keep2 & abn; q["guard_C3"] = keep3 & (abn | irr)
        return q

    def _guard(self, keep, score, prefer=None):
        keep = keep.copy(); n = len(keep)
        need = min(n, max(self.P["min_keep_frames"], int(np.ceil(self.P["min_keep_frac"] * n))))
        if keep.sum() < need:
            pref = np.zeros(n) if prefer is None else (~prefer).astype(float)
            for i in np.lexsort((score, pref)):
                if keep.sum() >= need:
                    break
                keep[i] = True
        return keep

    # ---- C4: khử nhiễu KHÔNG DỪNG, bảo toàn thành phần dừng -------------------
    def denoise_mbg(self, x):
        Z = librosa.stft(x, n_fft=self.P["stft_nfft"], hop_length=self.P["stft_hop"], window="hann")
        mag = np.abs(Z); med = np.median(mag, axis=1, keepdims=True)
        g = np.minimum(1.0, self.P["mbg_beta"] * med / (mag + 1e-12))
        return librosa.istft(Z * g, hop_length=self.P["stft_hop"], window="hann", length=len(x))

    # ---- C5: trừ phổ dừng (Boll 1979; Berouti et al. 1979), nhiễu = thống kê cực tiểu ----
    def denoise_ss(self, x):
        Z = librosa.stft(x, n_fft=self.P["stft_nfft"], hop_length=self.P["stft_hop"], window="hann")
        Pz = np.abs(Z) ** 2; q = self.P["ss_quantile"]
        noise = np.quantile(Pz, q, axis=1, keepdims=True) / (-np.log(1 - q))   # bù chệch phân vị (phân bố mũ)
        G = np.sqrt(np.maximum(1.0 - self.P["ss_alpha"] * noise / (Pz + 1e-20), self.P["ss_floor"] ** 2))
        return librosa.istft(Z * G, hop_length=self.P["stft_hop"], window="hann", length=len(x))

    # ---- C6: wavelet soft-threshold, ngưỡng phổ quát (VisuShrink) -------------
    def denoise_wt(self, x):
        w = pywt.Wavelet(self.P["wt_wavelet"])
        L = max(1, min(self.P["wt_level"], pywt.dwt_max_level(len(x), w.dec_len)))
        c = pywt.wavedec(x, w, level=L, mode="symmetric")
        sigma = np.median(np.abs(c[-1])) / 0.6745
        thr = sigma * np.sqrt(2 * np.log(len(x)))
        c = [c[0]] + [pywt.threshold(ci, thr, mode="soft") for ci in c[1:]]
        return pywt.waverec(c, w, mode="symmetric")[:len(x)]

    def signals(self, xr):
        """Trả về tín hiệu của mọi điều kiện + bảng QC khung + QC tệp."""
        P = self.P
        x1 = self.condition(xr)
        x2, spike_frac = self.spike_clip(x1)
        starts = self.frame_starts(len(xr))
        sig = {"C0_RAW": xr, "C1_BPF": x1, "C2_TR": x2, "C3_IR": x2}
        conds = P["conditions"]
        if "C4_MBG" in conds: sig["C4_MBG"] = self.denoise_mbg(x2)
        if "C5_SS" in conds: sig["C5_SS"] = self.denoise_ss(x2)
        if "C6_WT" in conds: sig["C6_WT"] = self.denoise_wt(x2)
        q = self.qc_frames(xr, x1, starts) if len(starts) else pd.DataFrame()
        return sig, starts, q, spike_frac

    def mask_for(self, cond, q):
        if cond in ("C0_RAW", "C1_BPF"):
            return np.ones(len(q), bool)
        if cond == "C2_TR":
            return q.keep_C2.values
        return q.keep_C3.values


def file_qc(xr, sr, P):
    """QC mức tệp (không nhìn nhãn): bão hoà, mất tín hiệu, DC, mức, SNR ước lượng."""
    n = len(xr); B = int(round(P["block_ms"] * sr / 1000)); nb = max(1, n // B)
    blk = 10 * np.log10((xr[:nb * B].reshape(nb, B) ** 2).mean(1) + 1e-20) if n >= B else np.array([-200.0])
    rms = np.sqrt(np.mean(xr ** 2)) + 1e-20
    d = dict(duration_s=n / sr, clip_frac=float((np.abs(xr) >= P["clip_level"]).mean()),
             zero_frac=float((xr == 0).mean()), dc_ratio=float(abs(xr.mean()) / rms),
             rms_dbfs=float(20 * np.log10(rms)), snr_proxy_db=float(np.percentile(blk, 95) - np.percentile(blk, 10)))
    reasons = []
    if n < int(P["frame_sec"] * sr): reasons.append("too_short")
    if d["clip_frac"] > P["qc_max_clip_frac"]: reasons.append("clipping")
    if d["zero_frac"] > P["qc_max_zero_frac"]: reasons.append("dropout")
    d["excluded"] = bool(reasons); d["reason"] = ",".join(reasons)
    return d


SR = int(cfg.target_sr) if cfg.target_sr else int(files.sr_native.max())
PARAMS = asdict(cfg)
CHAIN = SignalChain(SR, PARAMS)
print(f"sr = {SR} Hz | khung {CHAIN.F} mẫu ({cfg.frame_sec} s), bước {CHAIN.H} | dải phân tích "
      f"{cfg.f_lo:.0f}-{CHAIN.f_hi:.0f} Hz | {len(CHAIN.notches)} notch | {len(CHAIN.lsd_lo)} dải 1/3 octave cho LSD")

In [ ]:
# =========================================================
# 3.2 Hình 2-3: Minh hoạ chuỗi xử lý trên một bản ghi mỗi lớp
#     (các khung bị loại được tô màu theo lý do; phổ trung bình sau từng bộ khử nhiễu)
# =========================================================
def _pick_example(cls):
    cand = files[(files.cls == cls) & (files.duration_s >= 3 * cfg.frame_sec)]
    best, best_n = cand.iloc[0], -1
    for _, r in cand.head(10).iterrows():          # ưu tiên tệp có vài khung bị loại (minh hoạ rõ)
        xr = load_raw(r.path, SR); _, st_, q, _ = CHAIN.signals(xr)
        n_rej = int((~q.keep_C3).sum()) if len(q) else 0
        if 0 < n_rej <= len(q) // 2 and n_rej > best_n:
            best, best_n = r, n_rej
    return best


REASON_COLOR = {"f_event": CAT[3], "f_impulsive": CAT[7], "f_clip": CAT[6], "f_silence": "#9a9993",
                "f_spectral": CAT[4], "f_voiced": CAT[2]}
EX = {c: _pick_example(c) for c in ("leak", "noleak")}
fig = plt.figure(figsize=(12, 8.6))
gs = gridspec.GridSpec(4, 2, hspace=0.55, wspace=0.12, height_ratios=[0.8, 1, 1, 1])
psd_demo = {}
for j, c in enumerate(("noleak", "leak")):
    r = EX[c]; xr = load_raw(r.path, SR); sig, starts, q, _ = CHAIN.signals(xr)
    t = np.arange(len(xr)) / SR
    a0 = fig.add_subplot(gs[0, j]); a0.plot(t, xr, lw=0.3, color=INK2)
    for i, s in enumerate(starts):
        for rsn, col in REASON_COLOR.items():
            if q[rsn].iloc[i]:
                a0.axvspan(s / SR, (s + CHAIN.F) / SR, color=col, alpha=0.25, lw=0)
                break
    a0.set_title(f"({'ab'[j]}) {c}: {r.file_stem} - raw waveform, rejected frames shaded")
    a0.set_xlim(0, t[-1]); a0.set_xlabel("time (s)")
    for i, (cond, lab) in enumerate([("C1_BPF", "C1 band-pass + notch"), ("C4_MBG", "C4 median-background (proposed)"),
                                     ("C5_SS", "C5 spectral subtraction")]):
        if cond not in sig:
            continue
        ax = fig.add_subplot(gs[i + 1, j])
        S = librosa.amplitude_to_db(np.abs(librosa.stft(sig[cond], n_fft=1024, hop_length=256)), ref=np.max(np.abs(librosa.stft(sig["C1_BPF"], n_fft=1024, hop_length=256))))
        librosa.display.specshow(S, sr=SR, hop_length=256, x_axis="time", y_axis="log", ax=ax, cmap="magma", vmin=-80, vmax=0, rasterized=True)
        ax.set_ylim(cfg.f_lo, CHAIN.f_hi); ax.set_title(f"{lab}"); ax.set_xlabel("")
    psd_demo[c] = {k: sps.welch(v, fs=SR, nperseg=4096)[1] for k, v in sig.items()}
handles = [plt.Rectangle((0, 0), 1, 1, color=v, alpha=0.4) for v in REASON_COLOR.values()]
fig.legend(handles, [k[2:] for k in REASON_COLOR], loc="upper center", ncol=6, bbox_to_anchor=(0.5, 0.995))
savefig(fig, "Fig02_processing_examples", "Raw waveform with rejected frames and spectrograms after C1/C4/C5.")

fw = np.fft.rfftfreq(4096, 1 / SR)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for j, c in enumerate(("noleak", "leak")):
    for cond in [k for k in cfg.conditions if k in psd_demo[c] and k not in ("C2_TR", "C3_IR")]:
        ax[j].semilogx(fw[1:], 10 * np.log10(psd_demo[c][cond][1:] + 1e-20), color=COND_COLOR[cond], lw=1.2, label=cond)
    freq_axis(ax[j], cfg.f_lo, SR / 2); ax[j].set_title(f"({'ab'[j]}) {c}: long-term PSD after each denoiser")
ax[0].set_ylabel("PSD (dB re 1/Hz)"); ax[1].legend(ncol=2)
fig.tight_layout()
savefig(fig, "Fig03_denoiser_psd", "Long-term PSD of the example files after each processing condition.")

## 4. Trích xuất đặc trưng thủ công / Handcrafted feature extraction

Mỗi khung 2 s được mô tả bởi **~170 đặc trưng** thuộc 7 nhóm, đều đã được dùng trong các công trình về
nhận dạng rò rỉ bằng âm thanh/rung động và phân loại âm thanh môi trường:

| Nhóm | Đặc trưng | Ý nghĩa vật lý với rò rỉ | Tham chiếu |
|---|---|---|---|
| **Time-domain** | RMS, đỉnh, crest/impulse/shape/clearance factor, skewness, kurtosis, ZCR, Hjorth, entropy năng lượng, độ ổn định RMS khối, đường bao (CV, kurtosis), thời gian tương quan ACF, đỉnh chu kỳ ACF | ồn rò rỉ dừng, gần Gauss (kurtosis ≈ 3, crest thấp, đường bao phẳng); nhiễu xung thì ngược lại | Randall 2011; Hjorth 1970; Giannakopoulos 2015; Fares et al. 2023 |
| **Spectral shape** | centroid, spread, skewness, kurtosis, entropy, flatness, roll-off 50/85/95 %, độ dốc dB/octave, decrease, crest, tần số đỉnh, số đỉnh âm sắc, flux, spectral contrast (7 octave), spectral kurtosis | rò rỉ làm phổ "trắng" hơn/dịch lên trong dải phát xạ tia nước; nhiễu điện/bơm tạo đỉnh âm sắc | Peeters 2004; Jiang et al. 2002; Antoni 2006 |
| **Band energy** | năng lượng tương đối 25 dải 1/3 octave (25 Hz–6,3 kHz), 3 tỉ số dải, mức tuyệt đối 7 dải octave | **trả lời trực tiếp RQ1**: dải nào mang thông tin | Hunaidi & Chu 1999; Almeida et al. 2014 |
| **Cepstral** | MFCC 1–13 (mean, std), MFCC0, độ biến thiên delta, LFCC 1–13 | hình bao phổ, bất biến với hệ số khuếch đại (trừ c0) | Davis & Mermelstein 1980; Zhou et al. 2011 |
| **Wavelet** | năng lượng tương đối DWT (db4, 7 mức) và WPD (db4, mức 4, 16 dải 500 Hz), entropy wavelet | phân bố năng lượng đa phân giải — nhóm đặc trưng phổ biến nhất trong các bài báo rò rỉ ống | Mallat 1989; Coifman & Wickerhauser 1992; Xu et al. 2021 |
| **LPC** | 10 hệ số dự báo tuyến tính + sai số dự báo | mô hình cộng hưởng của ống/gậy nghe | Makhoul 1975; Cody, Dey & Narasimhan 2020 |
| **Complexity & modulation** | permutation entropy (trễ 1, 4), chiều fractal Higuchi, Katz, năng lượng điều biến đường bao 0,5–4 / 4–16 / 16–64 Hz | ồn rò rỉ phức tạp cao, không điều biến; tiếng nói điều biến ~4 Hz | Bandt & Pompe 2002; Higuchi 1988; Katz 1988; Atlas & Shamma 2003 |

Đặc trưng **phụ thuộc mức tín hiệu** (RMS, đỉnh, mức octave tuyệt đối, MFCC0) được đánh dấu `level_dep`
để kiểm tra riêng ở mục 10 (nếu hệ số khuếch đại của micro thay đổi giữa các lần thu, chúng có thể là confound).

In [ ]:
# =========================================================
# 4.1 Bộ trích xuất đặc trưng / Feature extractor
# =========================================================
def linear_filterbank(freqs, n, f_lo, f_hi):
    pts = np.linspace(f_lo, f_hi, n + 2); fb = np.zeros((n, len(freqs)))
    for i in range(n):
        l, c, r = pts[i], pts[i + 1], pts[i + 2]
        fb[i] = np.maximum(0, np.minimum((freqs - l) / (c - l), (r - freqs) / (r - c)))
    return fb


def perm_entropy(x, order=5, delay=1):
    emb = sliding_window_view(x, (order - 1) * delay + 1)[:, ::delay]
    codes = (np.argsort(emb, axis=1) * (order ** np.arange(order))).sum(1)
    _, cnt = np.unique(codes, return_counts=True)
    p = cnt / cnt.sum()
    return float(-(p * np.log(p)).sum() / np.log(math.factorial(order)))


def higuchi_fd(x, kmax=10):
    N = len(x); ks = np.arange(1, kmax + 1); lk = []
    for k in ks:
        Lm = [np.abs(np.diff(x[m::k])).sum() * (N - 1) / ((len(x[m::k]) - 1) * k) / k
              for m in range(k) if len(x[m::k]) > 1]
        lk.append(np.mean(Lm))
    return float(np.polyfit(np.log(1.0 / ks), np.log(np.asarray(lk) + 1e-20), 1)[0])


def katz_fd(x):
    dists = np.abs(np.diff(x)); L = dists.sum() + 1e-20
    a = np.log10(L / (dists.mean() + 1e-20)); dmax = np.max(np.abs(x - x[0])) + 1e-20
    return float(a / (a + np.log10(dmax / L)))


def _hz(v):
    return f"{v:g}" if v < 100 else f"{int(round(v))}"


class FeatureExtractor:
    def __init__(self, sr, P):
        self.sr = sr; nyq = sr / 2
        self.f_lo = P["f_lo"]; self.f_hi = min(P["f_hi"], 0.95 * nyq)
        self.nper = int(2 ** round(np.log2(0.256 * sr)))
        self.fw = np.fft.rfftfreq(self.nper, 1 / sr)
        self.m_an = (self.fw >= self.f_lo) & (self.fw <= self.f_hi)
        self.to_lo, self.to_hi, self.to_nom = third_octave_bands(self.f_lo, self.f_hi)
        oc = np.array([63, 125, 250, 500, 1000, 2000, 4000, 8000.0])
        self.oc = oc[oc * np.sqrt(2) <= self.f_hi]
        e = self.f_lo * 2 ** (np.arange(0, 400) / 6.0); e = e[e <= self.f_hi * 1.0001]
        self.fb_lo, self.fb_hi = e[:-1], e[1:]
        self.n_fft = int(2 ** round(np.log2(0.064 * sr))); self.hop = self.n_fft // 2
        self.sfreq = np.fft.rfftfreq(self.n_fft, 1 / sr)
        self.mel_fb = librosa.filters.mel(sr=sr, n_fft=self.n_fft, n_mels=40, fmin=self.f_lo, fmax=self.f_hi)
        self.lin_fb = linear_filterbank(self.sfreq, 40, self.f_lo, self.f_hi)
        ed = [0] + [62.5 * 2 ** i for i in range(7)]
        self.sc_names = [f"sc{i}_{_hz(ed[i])}-{_hz(ed[i + 1]) if i < 6 else 'nyq'}Hz" for i in range(7)]
        self.sk_nfft = int(2 ** round(np.log2(0.016 * sr))); self.fsk = np.fft.rfftfreq(self.sk_nfft, 1 / sr)
        self.dwt_L = max(2, int(np.floor(np.log2(nyq / 62.5))))
        self.dwt_names = [f"wd_A{self.dwt_L}_0-{_hz(nyq / 2 ** self.dwt_L)}Hz"] + \
            [f"wd_D{j}_{_hz(nyq / 2 ** j)}-{_hz(nyq / 2 ** (j - 1))}Hz" for j in range(self.dwt_L, 0, -1)]
        wbw = nyq / 16
        self.wp_names = [f"wp{i:02d}_{_hz(i * wbw)}-{_hz((i + 1) * wbw)}Hz" for i in range(16)]

    def __call__(self, x):
        sr = self.sr; x = np.asarray(x, np.float64); N = len(x); eps = 1e-12; d = {}
        xc = x - x.mean(); ax = np.abs(x)
        rms = np.sqrt(np.mean(x ** 2)) + eps; peak = ax.max() + eps; mav = ax.mean() + eps; sd = xc.std() + eps
        # ---------------- Time-domain ----------------
        d["td_rms_db"] = 20 * np.log10(rms)
        d["td_peak_db"] = 20 * np.log10(peak)
        d["td_crest"] = peak / rms
        d["td_impulse_factor"] = peak / mav
        d["td_shape_factor"] = rms / mav
        d["td_clearance_factor"] = peak / (np.mean(np.sqrt(ax)) ** 2 + eps)
        d["td_skewness"] = np.mean(xc ** 3) / sd ** 3
        d["td_kurtosis"] = np.mean(xc ** 4) / sd ** 4
        d["td_zcr"] = np.mean(np.signbit(x[1:]) != np.signbit(x[:-1]))
        dx = np.diff(xc); ddx = np.diff(dx)
        v0, v1, v2 = xc.var() + eps, dx.var() + eps, ddx.var() + eps
        mob = np.sqrt(v1 / v0)
        d["td_hjorth_mobility"] = mob
        d["td_hjorth_complexity"] = np.sqrt(v2 / v1) / mob
        n20 = N // 20; eb = (xc[:n20 * 20].reshape(20, n20) ** 2).sum(1) + eps; pe = eb / eb.sum()
        d["td_energy_entropy"] = -(pe * np.log(pe)).sum() / np.log(20)
        B = int(0.05 * sr); nb = N // B
        d["td_blockrms_std_db"] = (10 * np.log10((xc[:nb * B].reshape(nb, B) ** 2).mean(1) + eps)).std()
        env = np.abs(sps.hilbert(xc))
        d["td_env_cv"] = env.std() / (env.mean() + eps)
        d["td_env_kurtosis"] = sst.kurtosis(env, fisher=False)
        nfft = 1 << int(np.ceil(np.log2(2 * N)))
        r = np.fft.irfft(np.abs(np.fft.rfft(xc, nfft)) ** 2, nfft)[: int(0.05 * sr)]; r = r / (r[0] + eps)
        b_ = np.where(r < 1 / np.e)[0]; z_ = np.where(r < 0)[0]
        d["td_acf_decay_ms"] = (b_[0] if len(b_) else len(r)) / sr * 1000
        d["td_acf_zero_ms"] = (z_[0] if len(z_) else len(r)) / sr * 1000
        d["hm_acf_peak"] = r[int(sr / 500): int(sr / 50)].max()
        # ---------------- Spectral shape (Welch) ----------------
        f, Pxx = sps.welch(x, fs=sr, nperseg=self.nper, noverlap=self.nper // 2, window="hann", detrend="constant")
        fb, Pb = f[self.m_an], Pxx[self.m_an] + eps
        pn = Pb / Pb.sum(); c = (fb * pn).sum(); spread = np.sqrt(((fb - c) ** 2 * pn).sum()) + eps
        d["sp_centroid"] = c
        d["sp_spread"] = spread
        d["sp_skewness"] = ((fb - c) ** 3 * pn).sum() / spread ** 3
        d["sp_kurtosis"] = ((fb - c) ** 4 * pn).sum() / spread ** 4
        d["sp_entropy"] = -(pn * np.log(pn)).sum() / np.log(len(pn))
        d["sp_flatness"] = np.exp(np.mean(np.log(Pb))) / np.mean(Pb)
        cum = np.cumsum(pn)
        for q in (50, 85, 95):
            d[f"sp_rolloff{q}"] = fb[min(np.searchsorted(cum, q / 100), len(fb) - 1)]
        ldb = 10 * np.log10(Pb)
        d["sp_slope_db_oct"] = np.polyfit(np.log2(fb), ldb, 1)[0]
        d["sp_decrease"] = ((Pb[1:] - Pb[0]) / np.arange(1, len(Pb))).sum() / Pb[1:].sum()
        d["sp_crest_db"] = 10 * np.log10(Pb.max() / Pb.mean())
        d["sp_peak_freq"] = fb[np.argmax(Pb)]
        prom = ldb - sps.medfilt(ldb, 31)
        d["sp_n_tonal_peaks"] = len(sps.find_peaks(prom, height=10.0, distance=3)[0])
        d["sp_tonality_db"] = prom.max()
        # ---------------- Band energy ----------------
        E_tot = band_powers(f, Pxx, np.array([self.f_lo]), np.array([self.f_hi]))[0] + eps
        for nom, v in zip(self.to_nom, band_powers(f, Pxx, self.to_lo, self.to_hi)):
            d[f"be_{nom:g}Hz"] = 10 * np.log10(v / E_tot + eps)
        rr = band_powers(f, Pxx, np.array([self.f_lo, 1000.0, 100.0, 1000.0]),
                         np.array([1000.0, self.f_hi, 800.0, min(3000.0, self.f_hi)])) + eps
        d["br_below1k_vs_above_db"] = 10 * np.log10(rr[0] / rr[1])
        d["br_100-800Hz_frac"] = rr[2] / E_tot
        d["br_1k-3kHz_frac"] = rr[3] / E_tot
        for c_, v in zip(self.oc, band_powers(f, Pxx, self.oc / np.sqrt(2), self.oc * np.sqrt(2))):
            d[f"bl_oct{int(c_)}Hz_db"] = 10 * np.log10(v + eps)
        fine = band_powers(f, Pxx, self.fb_lo, self.fb_hi)
        # ---------------- Cepstral + flux + contrast (STFT 64 ms) ----------------
        S = np.abs(librosa.stft(x, n_fft=self.n_fft, hop_length=self.hop, center=False, window="hann"))
        Pw = S ** 2 + eps
        mf = dct(10 * np.log10(self.mel_fb @ Pw + eps), type=2, axis=0, norm="ortho")[:14]
        d["mfcc0_mean"] = mf[0].mean()
        for i in range(1, 14):
            d[f"mfcc{i}_mean"] = mf[i].mean()
        for i in range(1, 14):
            d[f"mfcc{i}_std"] = mf[i].std()
        d["mfcc_delta_std"] = np.diff(mf[1:], axis=1).std(1).mean()
        lf = dct(10 * np.log10(self.lin_fb @ Pw + eps), type=2, axis=0, norm="ortho")[:14]
        for i in range(1, 14):
            d[f"lfcc{i}_mean"] = lf[i].mean()
        Sn = S / (np.linalg.norm(S, axis=0, keepdims=True) + eps)
        fl = np.sqrt((np.diff(Sn, axis=1) ** 2).sum(0))
        d["sp_flux_mean"] = fl.mean()
        d["sp_flux_std"] = fl.std()
        con = librosa.feature.spectral_contrast(S=S, sr=sr, n_fft=self.n_fft, fmin=62.5, n_bands=6).mean(1)
        for nm, v in zip(self.sc_names, con):
            d[nm] = v
        # ---------------- Spectral kurtosis (Antoni 2006) ----------------
        A2 = np.abs(librosa.stft(xc, n_fft=self.sk_nfft, hop_length=self.sk_nfft // 4, center=False, window="hann")) ** 2
        sk = (A2 ** 2).mean(1) / (A2.mean(1) ** 2 + eps) - 2.0
        m = (self.fsk >= 100) & (self.fsk <= self.f_hi)
        d["sk_mean"] = sk[m].mean()
        d["sk_max"] = sk[m].max()
        d["sk_argmax_hz"] = self.fsk[m][np.argmax(sk[m])]
        for a, b, nm in [(100, 1000, "sk_100-1000Hz"), (1000, 3000, "sk_1000-3000Hz"), (3000, self.f_hi + 1, "sk_3000Hz-up")]:
            mm = (self.fsk >= a) & (self.fsk < b)
            d[nm] = sk[mm].mean() if mm.any() else np.nan
        # ---------------- Envelope modulation ----------------
        dec = max(1, int(sr / 250)); ne = N // dec
        e = env[:ne * dec].reshape(ne, dec).mean(1); e = e - e.mean()
        fm, Pm = sps.welch(e, fs=sr / dec, nperseg=min(256, ne))
        tot = Pm[fm >= 0.5].sum() + eps
        for a, b in [(0.5, 4), (4, 16), (16, 64)]:
            d[f"mod_{a:g}-{b:g}Hz"] = Pm[(fm >= a) & (fm < b)].sum() / tot
        # ---------------- Wavelet ----------------
        cw = pywt.wavedec(xc, "db4", level=self.dwt_L, mode="symmetric")
        ew = np.array([np.sum(ci ** 2) for ci in cw]) + eps; ew /= ew.sum()
        for nm, v in zip(self.dwt_names, ew):
            d[nm] = v
        d["wd_entropy"] = -(ew * np.log(ew)).sum() / np.log(len(ew))
        wp = pywt.WaveletPacket(xc, "db4", mode="symmetric", maxlevel=4)
        ep = np.array([np.sum(nd.data ** 2) for nd in wp.get_level(4, order="freq")]) + eps; ep /= ep.sum()
        for nm, v in zip(self.wp_names, ep):
            d[nm] = v
        d["wp_entropy"] = -(ep * np.log(ep)).sum() / np.log(len(ep))
        # ---------------- Complexity ----------------
        d["nl_perm_entropy_d1"] = perm_entropy(xc, 5, 1)
        d["nl_perm_entropy_d4"] = perm_entropy(xc, 5, 4)
        d["nl_higuchi_fd"] = higuchi_fd(xc, 10)
        d["nl_katz_fd"] = katz_fd(xc)
        # ---------------- LPC ----------------
        try:
            a = librosa.lpc(xc / sd, order=10)
            for i in range(1, 11):
                d[f"lpc{i}"] = a[i]
            d["lpc_err_db"] = 10 * np.log10(sps.lfilter(a, [1.0], xc / sd).var() + eps)
        except Exception:
            for i in range(1, 11):
                d[f"lpc{i}"] = np.nan
            d["lpc_err_db"] = np.nan
        return d, fine.astype(np.float32), Pxx


FX = FeatureExtractor(SR, PARAMS)
_d, _fine, _p = FX(np.random.default_rng(0).standard_normal(CHAIN.F) * 0.01)
FEATURES = list(_d.keys())
print(f"{len(FEATURES)} đặc trưng / khung | {len(_fine)} dải 1/6 octave cho phân tích dải tần (E1) | "
      f"Welch nperseg={FX.nper} (Δf = {SR / FX.nper:.2f} Hz)")

In [ ]:
# =========================================================
# 4.2 Danh mục đặc trưng (nhóm, mô tả, dải tần, phụ thuộc mức) - Bảng 3 của bài báo
# =========================================================
GROUPS = ["Time-domain", "Spectral shape", "Band energy", "Cepstral", "Wavelet", "LPC", "Complexity & modulation"]
GROUP_COLOR = {g: CAT[i] for i, g in enumerate(GROUPS)}
_FAM = [("td_", "Time-domain statistics", "Time-domain"), ("hm_", "Periodicity (ACF)", "Time-domain"),
        ("sp_", "Spectral shape", "Spectral shape"), ("sc", "Spectral contrast", "Spectral shape"),
        ("sk_", "Spectral kurtosis", "Spectral shape"), ("be_", "Relative 1/3-octave band energy", "Band energy"),
        ("br_", "Band-energy ratio", "Band energy"), ("bl_", "Absolute octave-band level", "Band energy"),
        ("mfcc", "MFCC", "Cepstral"), ("lfcc", "LFCC", "Cepstral"), ("wd_", "DWT relative energy", "Wavelet"),
        ("wp", "WPD relative energy", "Wavelet"), ("lpc", "Linear prediction", "LPC"),
        ("nl_", "Nonlinear complexity", "Complexity & modulation"), ("mod_", "Envelope modulation", "Complexity & modulation")]
_DESC = {
    "td_rms_db": "RMS level (dBFS)", "td_peak_db": "peak level (dBFS)", "td_crest": "crest factor peak/RMS",
    "td_impulse_factor": "impulse factor peak/mean|x|", "td_shape_factor": "shape factor RMS/mean|x|",
    "td_clearance_factor": "clearance (margin) factor", "td_skewness": "skewness", "td_kurtosis": "kurtosis (Gauss = 3)",
    "td_zcr": "zero-crossing rate", "td_hjorth_mobility": "Hjorth mobility", "td_hjorth_complexity": "Hjorth complexity",
    "td_energy_entropy": "entropy of sub-block energy (stationarity)", "td_blockrms_std_db": "std of 50-ms block level",
    "td_env_cv": "Hilbert envelope coefficient of variation", "td_env_kurtosis": "Hilbert envelope kurtosis",
    "td_acf_decay_ms": "ACF 1/e decay time", "td_acf_zero_ms": "ACF first zero crossing", "hm_acf_peak": "max ACF at 2-20 ms lag (periodicity)",
    "sp_centroid": "spectral centroid", "sp_spread": "spectral spread (bandwidth)", "sp_skewness": "spectral skewness",
    "sp_kurtosis": "spectral kurtosis (shape)", "sp_entropy": "spectral entropy", "sp_flatness": "spectral flatness (Wiener entropy)",
    "sp_rolloff50": "median frequency (50% roll-off)", "sp_rolloff85": "85% roll-off", "sp_rolloff95": "95% roll-off",
    "sp_slope_db_oct": "spectral slope (dB/octave)", "sp_decrease": "spectral decrease", "sp_crest_db": "spectral crest",
    "sp_peak_freq": "dominant frequency", "sp_n_tonal_peaks": "number of tonal peaks (>10 dB)", "sp_tonality_db": "max peak prominence",
    "sp_flux_mean": "mean spectral flux", "sp_flux_std": "std of spectral flux", "sk_mean": "mean spectral kurtosis 100 Hz-f_hi",
    "sk_max": "max spectral kurtosis", "sk_argmax_hz": "frequency of max spectral kurtosis",
    "br_below1k_vs_above_db": "energy below/above 1 kHz (dB)", "br_100-800Hz_frac": "energy fraction 100-800 Hz",
    "br_1k-3kHz_frac": "energy fraction 1-3 kHz", "mfcc0_mean": "MFCC c0 (log energy)", "mfcc_delta_std": "mean std of delta-MFCC",
    "wd_entropy": "wavelet energy entropy (DWT)", "wp_entropy": "wavelet packet energy entropy",
    "nl_perm_entropy_d1": "permutation entropy (m=5, lag 1)", "nl_perm_entropy_d4": "permutation entropy (m=5, lag 4)",
    "nl_higuchi_fd": "Higuchi fractal dimension", "nl_katz_fd": "Katz fractal dimension", "lpc_err_db": "LPC prediction error (dB)",
}
LEVEL_DEP = {n for n in FEATURES if n in ("td_rms_db", "td_peak_db", "mfcc0_mean") or n.startswith("bl_")}


def feature_band(n):
    m = re.search(r"_(\d+(?:\.\d+)?)Hz$", n)
    if n.startswith("be_") and m:
        c = float(m.group(1)); cc = 1000 * 2 ** (np.round(3 * np.log2(c / 1000)) / 3)
        return cc * 2 ** (-1 / 6), cc * 2 ** (1 / 6)
    m = re.search(r"oct(\d+)Hz", n)
    if m:
        c = float(m.group(1)); return c / np.sqrt(2), c * np.sqrt(2)
    m = re.search(r"_(\d+(?:\.\d+)?)-(\d+(?:\.\d+)?|nyq)Hz", n)
    if m and not n.startswith("mod_"):
        lo = float(m.group(1)); hi = SR / 2 if m.group(2) == "nyq" else float(m.group(2))
        return max(lo, cfg.f_lo), hi
    if n == "sk_3000Hz-up":
        return 3000.0, CHAIN.f_hi
    return np.nan, np.nan


def feature_meta(n):
    fam, grp = next(((f, g) for p, f, g in _FAM if n.startswith(p)), ("Other", "Time-domain"))
    desc = _DESC.get(n)
    if desc is None:
        m = re.match(r"(mfcc|lfcc)(\d+)_(mean|std)", n)
        if m:
            desc = f"{m.group(1).upper()} c{m.group(2)} {m.group(3)}"
        elif n.startswith("be_"):
            desc = f"relative energy, 1/3-octave band centred {n[3:]}"
        elif n.startswith("bl_"):
            desc = f"absolute level, octave band {n[6:].replace('_db', '')}"
        elif n.startswith(("wd_", "wp")):
            desc = f"relative wavelet energy {n.split('_', 1)[1]}"
        elif n.startswith("sc"):
            desc = f"spectral contrast {n.split('_', 1)[1]}"
        elif n.startswith("sk_"):
            desc = f"mean spectral kurtosis {n[3:]}"
        elif n.startswith("mod_"):
            desc = f"relative envelope modulation energy {n[4:]}"
        elif n.startswith("lpc"):
            desc = f"LPC coefficient a{n[3:]}"
        else:
            desc = n
    lo, hi = feature_band(n)
    return dict(feature=n, family=fam, group=grp, description=desc, level_dep=n in LEVEL_DEP, band_lo=lo, band_hi=hi)


FEAT_META = pd.DataFrame([feature_meta(n) for n in FEATURES]).set_index("feature")
T03 = FEAT_META.reset_index().groupby(["group", "family"]).agg(n=("feature", "size"),
                                                              examples=("feature", lambda s: ", ".join(list(s)[:4]))).reset_index()
display(Markdown(f"**Bảng 3. Danh mục {len(FEATURES)} đặc trưng thủ công.**")); display(T03)
savetab(T03, "T03_feature_catalogue_summary"); savetab(FEAT_META.reset_index(), "T03b_feature_catalogue_full")
print("Đặc trưng phụ thuộc mức tín hiệu:", sorted(LEVEL_DEP))

In [ ]:
# =========================================================
# 4.3 Chạy chuỗi xử lý + trích đặc trưng cho MỌI tệp và MỌI điều kiện (song song, có cache)
# =========================================================
FEATURE_VERSION = "ml-v1.0"


def process_file(i, path, sr, P):
    t0 = time.time()
    chain, fx = SignalChain(sr, P), FeatureExtractor(sr, P)
    xr = load_raw(path, sr)
    qc = file_qc(xr, sr, P)
    out = dict(i=i, qc=qc, frames=None, feats={}, fine={}, psd={}, time=0.0, n_feat_frames=0)
    if qc["excluded"]:
        return out
    sig, starts, q, spike = chain.signals(xr)
    qc["spike_frac"] = spike
    q.insert(0, "frame_idx", np.arange(len(starts))); q.insert(1, "t_start", starts / sr)
    out["frames"] = q
    for cond in P["conditions"]:
        keep = np.where(chain.mask_for(cond, q))[0]; x = sig[cond]
        rows, fines, psum = [], [], 0.0
        for fi in keep:
            dd, fine, pxx = fx(x[starts[fi]: starts[fi] + chain.F])
            rows.append(dd); fines.append(fine); psum = psum + pxx
        out["feats"][cond] = (keep, rows)
        out["fine"][cond] = np.asarray(fines, np.float32)
        out["psd"][cond] = (psum, len(rows))
        out["n_feat_frames"] += len(rows)
    out["time"] = time.time() - t0
    return out


def _fingerprint():
    h = hashlib.sha1(FEATURE_VERSION.encode())
    for p in sorted(files.path):
        st = os.stat(p); h.update(f"{p}|{st.st_size}|{int(st.st_mtime)}".encode())
    keys = ["target_sr", "frame_sec", "hop_sec", "f_lo", "f_hi", "hp_hz", "lp_hz", "filt_order", "notch_hz",
            "notch_harmonics", "notch_q", "clip_level", "qc_max_clip_frac", "qc_max_zero_frac", "spike_k", "block_ms",
            "z_thr", "event_db", "kurt_thr", "frame_clip_frac", "lsd_z_thr", "lsd_min_db", "voicing_thr",
            "voicing_frac_thr", "voicing_excess", "min_keep_frac", "min_keep_frames", "stft_nfft", "stft_hop",
            "mbg_beta", "ss_alpha", "ss_floor", "ss_quantile", "wt_wavelet", "wt_level", "conditions"]
    h.update(json.dumps({k: PARAMS[k] for k in keys}, sort_keys=True, default=str).encode())
    return h.hexdigest()[:12]


_cache = CACHE_DIR / f"features_{_fingerprint()}.pkl"
if _cache.exists():
    RESULTS = joblib.load(_cache); print(f"Nạp đặc trưng từ cache: {_cache.name}")
else:
    t0 = time.time()
    RESULTS = Parallel(n_jobs=N_JOBS, backend="loky", verbose=0)(
        delayed(process_file)(i, r.path, SR, PARAMS) for i, r in files.iterrows())
    joblib.dump(RESULTS, _cache, compress=3)
    _nf = sum(r["n_feat_frames"] for r in RESULTS)
    print(f"Trích đặc trưng xong trong {time.time() - t0:.0f} s ({_nf} khung x điều kiện; "
          f"{sum(r['time'] for r in RESULTS) / max(_nf, 1) * 1000:.1f} ms CPU/khung kể cả xử lý) -> {_cache.name}")

# ---- Lắp bảng đặc trưng theo điều kiện --------------------------------------
META_COLS = ["record_id", "file_stem", "label", "cls", "split", "frame_idx", "t_start"]
FEAT, FINE, PSD_REC, QC_FRAMES, QC_FILES = {}, {}, {}, [], []
for cond in cfg.conditions:
    parts, fparts = [], []
    for res in RESULTS:
        if cond not in res["feats"] or not len(res["feats"][cond][1]):
            continue
        r = files.loc[res["i"]]; keep, rows = res["feats"][cond]
        df = pd.DataFrame(rows, columns=FEATURES)
        meta = pd.DataFrame({"record_id": r.record_id, "file_stem": r.file_stem, "label": r.label, "cls": r.cls,
                             "split": r.split, "frame_idx": keep, "t_start": keep * cfg.hop_sec})
        parts.append(pd.concat([meta, df], axis=1)); fparts.append(res["fine"][cond])
        ps, n = res["psd"][cond]
        a = PSD_REC.setdefault(cond, {}).setdefault(r.record_id, [0.0, 0])
        a[0] = a[0] + ps; a[1] += n
    FEAT[cond] = pd.concat(parts, ignore_index=True)
    FEAT[cond][FEATURES] = FEAT[cond][FEATURES].replace([np.inf, -np.inf], np.nan).astype(np.float32)
    FINE[cond] = np.vstack(fparts)
for res in RESULTS:
    r = files.loc[res["i"]]
    QC_FILES.append({"file_stem": r.file_stem, "record_id": r.record_id, "cls": r.cls, **res["qc"]})
    if res["frames"] is not None:
        QC_FRAMES.append(res["frames"].assign(file_stem=r.file_stem, record_id=r.record_id, cls=r.cls))
QC_FILES = pd.DataFrame(QC_FILES); QC_FRAMES = pd.concat(QC_FRAMES, ignore_index=True)
REC_KEPT = sorted(FEAT["C0_RAW"].record_id.unique())
lost = sorted(set(records.record_id) - set(REC_KEPT))
if lost:
    print(f"!! {len(lost)} bản ghi mất toàn bộ tệp do QC mức tệp: {lost} -> loại khỏi phân tích.")
records = records[records.record_id.isin(REC_KEPT)].reset_index(drop=True)
for cond in cfg.conditions:
    assert set(FEAT[cond].record_id) == set(REC_KEPT), f"{cond}: thiếu bản ghi sau khi lọc khung"
print(pd.DataFrame({c: [len(FEAT[c]), FEAT[c].record_id.nunique()] for c in cfg.conditions},
                   index=["frames", "records"]).T)

In [ ]:
# =========================================================
# 4.4 Bảng 2 + Hình 4: QC mức tệp và tỉ lệ khung bị loại theo lý do / lớp
#     (kiểm tra confound: nếu một lớp bị loại nhiều hơn hẳn thì chính bộ lọc có thể tạo ra khác biệt)
# =========================================================
excl = QC_FILES[QC_FILES.excluded]
print(f"Tệp bị loại ở QC mức tệp: {len(excl)}/{len(QC_FILES)}", excl[["file_stem", "cls", "reason"]].to_dict("records") if len(excl) else "")
REASONS = ["f_clip", "f_silence", "f_event", "f_impulsive", "f_spectral", "f_voiced"]
rej = (QC_FRAMES.groupby("cls")[REASONS + ["keep_C2", "keep_C3"]].mean()
       .assign(rejected_C2=lambda d: 1 - d.keep_C2, rejected_C3=lambda d: 1 - d.keep_C3)
       .drop(columns=["keep_C2", "keep_C3"]).T.round(4))
rec_rej = QC_FRAMES.groupby(["record_id", "cls"]).keep_C3.mean().rsub(1).reset_index(name="rej_rate")
p_rej = mwu_p(rec_rej.loc[rec_rej.cls == "leak", "rej_rate"], rec_rej.loc[rec_rej.cls == "noleak", "rej_rate"])
d_rej = cliffs_delta(rec_rej.loc[rec_rej.cls == "leak", "rej_rate"], rec_rej.loc[rec_rej.cls == "noleak", "rej_rate"])
display(Markdown("**Bảng 2. Tỉ lệ khung bị gắn cờ theo lý do và theo lớp (mức khung).**")); display(rej)
savetab(rej.reset_index().rename(columns={"index": "reason"}), "T02_rejection_by_reason")
savetab(QC_FILES, "T02b_file_qc")
print(f"Tỉ lệ loại (C3) mức bản ghi: leak trung vị {rec_rej[rec_rej.cls == 'leak'].rej_rate.median():.3f} | "
      f"noleak {rec_rej[rec_rej.cls == 'noleak'].rej_rate.median():.3f} | Mann-Whitney p = {p_rej:.3f}, Cliff δ = {d_rej:+.2f}")
REJ_CONFOUND = bool(np.isfinite(p_rej) and p_rej < 0.05)
print("=> " + ("!! Tỉ lệ loại KHÁC NHAU giữa hai lớp: phải thảo luận như một confound tiềm ẩn."
               if REJ_CONFOUND else "Tỉ lệ loại không khác biệt có ý nghĩa giữa hai lớp -> bộ lọc không tạo khác biệt giả."))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1.6, 1]})
xx = np.arange(len(REASONS) + 2); labels = [r[2:] for r in REASONS] + ["total C2", "total C3"]
for j, c in enumerate(["noleak", "leak"]):
    v = rej.loc[REASONS + ["rejected_C2", "rejected_C3"], c].values * 100
    ax[0].bar(xx + (j - 0.5) * 0.38, v, width=0.36, color=PAL[c], label=c)
ax[0].set_xticks(xx); ax[0].set_xticklabels(labels, rotation=25); ax[0].set_ylabel("% of frames flagged")
ax[0].set_title("(a) Frame rejection by reason"); ax[0].legend()
for j, c in enumerate(["noleak", "leak"]):
    v = rec_rej.loc[rec_rej.cls == c, "rej_rate"].values * 100
    ax[1].scatter(np.full(len(v), j) + np.random.default_rng(j).uniform(-0.12, 0.12, len(v)), v, s=14,
                  color=PAL[c], edgecolor="white", linewidth=0.5)
    ax[1].hlines(np.median(v), j - 0.25, j + 0.25, color=INK, lw=1.5)
ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(["noleak", "leak"]); ax[1].set_ylabel("% frames removed (C3)")
ax[1].set_title(f"(b) Per-recording removal rate (MWU p = {p_rej:.2f})")
fig.tight_layout()
savefig(fig, "Fig04_rejection_stats", "Frame rejection statistics by reason and class.")

## 5. E1 — Dải tần nào chứa tín hiệu rò rỉ? / Which frequency band carries the leak signal? (RQ1)

Ba lớp bằng chứng độc lập, đều ở **mức bản ghi** (đơn vị độc lập), trên điều kiện `cfg.e1_condition`
(đã loại xung và âm thanh không liên quan, **chưa** khử nhiễu để không làm méo phổ):

1. **E1a – Phổ trung bình dài hạn (LTAS)** theo lớp và **phổ hiệu** leak − noleak (dB) kèm CI bootstrap.
2. **E1b – Phổ khả năng phân biệt**: AUC đơn biến của năng lượng tương đối trong từng dải 1/6 octave,
   kiểm định Mann–Whitney + hiệu chỉnh Benjamini–Hochberg (FDR) và Holm; bảng 1/3 octave (Bảng 4).
3. **E1d – Quét dải bằng mô hình**: huấn luyện Random Forest chỉ với thông tin trong một cửa sổ tần số
   (octave trượt 1/3 octave; thông thấp tích luỹ `[f_lo, fc]`; thông cao tích luỹ `[fc, f_hi]`), đánh giá
   bằng CV lặp nhóm theo bản ghi. **Dải thiết yếu** = `[fc_HP*, fc_LP*]` trong đó `fc_LP*` là tần số cắt
   thông thấp nhỏ nhất và `fc_HP*` là tần số cắt thông cao lớn nhất mà AUC còn ≥ AUC(toàn dải) − `e1_tol`.

> E1 là phân tích **mô tả**: không một tham số nào của E2–E4 được chọn từ kết quả E1 (dải lọc C1 và
> danh mục đặc trưng được cố định trước). Vì vậy có thể dùng toàn bộ bản ghi (`cfg.e1_scope = "all"`)
> để có độ mạnh thống kê lớn nhất mà không làm rò rỉ thông tin sang tập kiểm tra.

In [ ]:
# =========================================================
# 5.1 Chuẩn bị: kế hoạch fold mức bản ghi dùng chung cho mọi thí nghiệm (thiết kế GHÉP CẶP)
# =========================================================
def make_record_folds(rec_ids, rec_y, n_splits, seed):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [np.asarray(rec_ids)[te] for _, te in skf.split(np.zeros(len(rec_ids)), rec_y)]


REC_LABEL = records.set_index("record_id").label
SCOPE_RECS = {"dev": records.loc[records.split == "dev", "record_id"].values,
              "all": records.record_id.values, "test": records.loc[records.split == "test", "record_id"].values}
FOLDS = {}
for scope in ("dev", "all"):
    ids = np.sort(SCOPE_RECS[scope]); yy = REC_LABEL.loc[ids].values
    k = int(min(cfg.n_splits, np.bincount(yy).min()))
    for s in sorted(set(cfg.seeds) | set(cfg.e1_seeds)):
        FOLDS[(scope, s)] = make_record_folds(ids, yy, k, s)
print({k: [len(f) for f in v] for k, v in list(FOLDS.items())[:2]}, "... (số bản ghi test mỗi fold)")


def sample_weights(y, g):
    """Mỗi bản ghi tổng trọng số như nhau; mỗi lớp tổng trọng số như nhau."""
    y = np.asarray(y); g = np.asarray(g); w = np.ones(len(y), float)
    if cfg.record_balanced:
        _, inv, cnt = np.unique(g, return_inverse=True, return_counts=True); w = 1.0 / cnt[inv]
    for c in (0, 1):
        m = y == c
        if m.any():
            w[m] *= 0.5 / w[m].sum()
    return w * len(w) / w.sum()


def rec_scores(scores, g, rec_ids):
    s = pd.Series(np.asarray(scores, float)).groupby(np.asarray(g)).mean()
    return s.reindex(rec_ids).values


E1_COND = cfg.e1_condition if cfg.e1_condition in FEAT else cfg.conditions[0]
E1_RECS = np.sort(SCOPE_RECS[cfg.e1_scope])
_e1 = FEAT[E1_COND]; _m = _e1.record_id.isin(E1_RECS).values
E1_DF = _e1.loc[_m].reset_index(drop=True); E1_FINE = FINE[E1_COND][_m]
E1_Y = REC_LABEL.loc[E1_RECS].values
FB_FC = np.sqrt(FX.fb_lo * FX.fb_hi)
print(f"E1 trên {len(E1_RECS)} bản ghi ({E1_Y.sum()} leak / {(1 - E1_Y).sum()} noleak), điều kiện {E1_COND}, "
      f"{len(E1_DF)} khung, {E1_FINE.shape[1]} dải 1/6 octave")

In [ ]:
# =========================================================
# 5.2 E1a + E1b: LTAS, phổ hiệu và phổ khả năng phân biệt (mức bản ghi)
# =========================================================
fw = FX.fw
_man = FX.m_an
REC_PSD = np.vstack([PSD_REC[E1_COND][r][0] / PSD_REC[E1_COND][r][1] for r in E1_RECS])
REL_PSD_DB = 10 * np.log10(REC_PSD[:, _man] / REC_PSD[:, _man].sum(1, keepdims=True) + 1e-20)
f_an = fw[_man]

# năng lượng 1/6 octave mức bản ghi (trung bình tuyến tính các khung) -> tương đối -> dB
_fb = pd.DataFrame(E1_FINE).groupby(E1_DF.record_id.values).mean().reindex(E1_RECS).values
FB_REL_DB = 10 * np.log10(_fb / _fb.sum(1, keepdims=True) + 1e-20)
FB_ABS_DB = 10 * np.log10(_fb + 1e-20)

rng = np.random.default_rng(0)
pos, neg = np.where(E1_Y == 1)[0], np.where(E1_Y == 0)[0]
nb_ = max(200, cfg.n_boot // 2)
delta = np.median(FB_REL_DB[pos], 0) - np.median(FB_REL_DB[neg], 0)
boots = np.array([np.median(FB_REL_DB[rng.choice(pos, len(pos))], 0) - np.median(FB_REL_DB[rng.choice(neg, len(neg))], 0)
                  for _ in range(nb_)])
d_lo, d_hi = np.percentile(boots, [2.5, 97.5], axis=0)

rows = []
for j in range(FB_REL_DB.shape[1]):
    a = auc_safe(E1_Y, FB_REL_DB[:, j]); lo, hi = boot_auc_ci(E1_Y, FB_REL_DB[:, j], n_boot=nb_, seed=j)
    rows.append(dict(f_lo=FX.fb_lo[j], f_hi=FX.fb_hi[j], fc=FB_FC[j], auc_rel=a, auc_lo=lo, auc_hi=hi,
                     auc_abs=auc_safe(E1_Y, FB_ABS_DB[:, j]), delta_db=delta[j], delta_lo=d_lo[j], delta_hi=d_hi[j],
                     p=mwu_p(FB_REL_DB[pos, j], FB_REL_DB[neg, j])))
E1B = pd.DataFrame(rows)
E1B["q_bh"] = bh_fdr(E1B.p.values); E1B["p_holm"] = holm(E1B.p.values)
E1B["sep"] = (E1B.auc_rel - 0.5).abs() * 2        # sức phân biệt 0..1, không phụ thuộc chiều
savetab(E1B.round(5), "T04b_fine_band_auc")

fig, ax = plt.subplots(3, 1, figsize=(9, 8.4), sharex=True)
for c, yv in (("noleak", 0), ("leak", 1)):
    A = REL_PSD_DB[E1_Y == yv]; med = np.median(A, 0); q1, q3 = np.percentile(A, [25, 75], axis=0)
    k = 9; ker = np.ones(k) / k
    sm = lambda v: np.convolve(v, ker, mode="same")
    ax[0].plot(f_an, sm(med), color=PAL[c], lw=1.4, label=f"{c} (n={len(A)})")
    ax[0].fill_between(f_an, sm(q1), sm(q3), color=PAL[c], alpha=0.18, lw=0)
_inb = f_an <= min(cfg.lp_hz, CHAIN.f_hi) * 0.97
_lvl = np.median(REL_PSD_DB[:, _inb], 0)
ax[0].set_ylim(np.percentile(_lvl, 0.5) - 8, np.percentile(_lvl, 99.5) + 5)
ax[0].set_ylabel("relative PSD (dB re total)"); ax[0].legend(loc="lower left")
ax[0].set_title(f"(a) Long-term average spectrum per class, median and IQR over recordings [{E1_COND}]")
ax[1].axhline(0, color=INK2, lw=0.8)
ax[1].fill_between(FB_FC, E1B.delta_lo, E1B.delta_hi, color=PAL["leak"], alpha=0.2, lw=0, step="mid")
ax[1].step(FB_FC, E1B.delta_db, where="mid", color=PAL["leak"], lw=1.5)
ax[1].set_ylabel("Δ level leak − noleak (dB)")
ax[1].set_title("(b) Difference spectrum (1/6 octave) with 95% bootstrap CI")
ax[2].axhline(0.5, color=INK2, lw=0.8)
ax[2].fill_between(FB_FC, E1B.auc_lo, E1B.auc_hi, color=CAT[0], alpha=0.18, lw=0, step="mid")
ax[2].step(FB_FC, E1B.auc_rel, where="mid", color=CAT[0], lw=1.5, label="relative band energy")
ax[2].step(FB_FC, E1B.auc_abs, where="mid", color=CAT[2], lw=1.1, label="absolute band level")
sig_ = E1B.q_bh < 0.05
ax[2].scatter(FB_FC[sig_], E1B.auc_rel[sig_], s=22, color=CAT[0], edgecolor="white", zorder=3, label="q(BH) < 0.05")
ax[2].set_ylabel("record-level AUC"); ax[2].set_xlabel("frequency (Hz)"); ax[2].legend(loc="best", ncol=3)
ax[2].set_title("(c) Univariate discriminability per 1/6-octave band (AUC > 0.5: higher in leak)")
freq_axis(ax[2], cfg.f_lo, CHAIN.f_hi)
fig.tight_layout()
savefig(fig, "Fig05_ltas_discriminability", "LTAS per class, difference spectrum and per-band AUC.")

# ---- Bảng 4: dải 1/3 octave (đặc trưng be_*) --------------------------------
_be = [n for n in FEATURES if n.startswith("be_")]
_rm = E1_DF.groupby("record_id")[_be].mean().reindex(E1_RECS)
T04 = []
for n in _be:
    v = _rm[n].values; lo_, hi_ = feature_band(n)
    a = auc_safe(E1_Y, v); ci = boot_auc_ci(E1_Y, v, n_boot=nb_, seed=1)
    T04.append(dict(band=n[3:], f_lo=round(lo_, 1), f_hi=round(hi_, 1), leak_median_db=np.median(v[pos]),
                    noleak_median_db=np.median(v[neg]), auc=a, auc_lo=ci[0], auc_hi=ci[1],
                    cliffs_delta=cliffs_delta(v[pos], v[neg]), p=mwu_p(v[pos], v[neg])))
T04 = pd.DataFrame(T04); T04["p_holm"] = holm(T04.p.values); T04["q_bh"] = bh_fdr(T04.p.values)
display(Markdown("**Bảng 4. Năng lượng tương đối theo dải 1/3 octave — AUC mức bản ghi (AUC > 0,5: leak cao hơn).**"))
display(T04.round(4)); savetab(T04.round(5), "T04_third_octave_auc")
_top = T04.assign(sep=(T04.auc - 0.5).abs()).sort_values("sep", ascending=False).head(5)
print("5 dải 1/3 octave phân biệt mạnh nhất:", "; ".join(
    f"{r.band} (AUC {r.auc:.2f}, p_Holm {r.p_holm:.3f})" for r in _top.itertuples()))

In [ ]:
# =========================================================
# 5.3 E1d: Quét dải tần bằng mô hình (Random Forest, CV lặp nhóm theo bản ghi)
# =========================================================
def _win_idx(a, b):
    return np.where((FX.fb_lo >= a * 0.999) & (FX.fb_hi <= b * 1.001))[0]


WINDOWS = []
_c = cfg.f_lo
while 2 * _c <= CHAIN.f_hi * 1.001:
    WINDOWS.append(("octave", _c, 2 * _c)); _c *= 2 ** (1 / 3)
for fc in [125, 250, 500, 750, 1000, 1500, 2000, 3000, 4000, 5000, 6000, CHAIN.f_hi]:
    if cfg.f_lo < fc <= CHAIN.f_hi:
        WINDOWS.append(("lowpass", cfg.f_lo, fc))
for fc in [cfg.f_lo, 125, 250, 500, 1000, 1500, 2000, 3000, 4000, 5000]:
    if fc < CHAIN.f_hi / 1.5:
        WINDOWS.append(("highpass", fc, CHAIN.f_hi))
WINDOWS = [(k, a, b, _win_idx(a, b)) for k, a, b in WINDOWS]
WINDOWS = [w for w in WINDOWS if len(w[3]) >= 2]
E1_LB = 10 * np.log10(E1_FINE.astype(np.float64) + 1e-20)


def _win_features(LB, idx, shape_only=False):
    lin = 10 ** (LB[:, idx] / 10); tot = 10 * np.log10(lin.sum(1, keepdims=True) + 1e-20)
    rel = LB[:, idx] - tot
    return rel if shape_only else np.hstack([rel, tot])


def e1_fold_job(LBtr, ytr, gtr, LBte, windows, seed):
    w = sample_weights(ytr, gtr); out = {}
    for wi, (kind, a, b, idx) in enumerate(windows):
        for shape_only in ((False, True) if kind == "octave" else (False,)):
            m = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_features="sqrt", n_jobs=1, random_state=seed)
            m.fit(_win_features(LBtr, idx, shape_only), ytr, sample_weight=w)
            out[(wi, shape_only)] = m.predict_proba(_win_features(LBte, idx, shape_only))[:, 1].astype(np.float32)
    return out


_g = E1_DF.record_id.values; _yf = E1_DF.label.values
jobs = []
for s in cfg.e1_seeds:
    for fi, te_r in enumerate(FOLDS[(cfg.e1_scope, s)]):
        te = np.isin(_g, te_r); jobs.append((s, te))
t0 = time.time()
_res = Parallel(n_jobs=N_JOBS, backend="loky")(
    delayed(e1_fold_job)(E1_LB[~te], _yf[~te], _g[~te], E1_LB[te], WINDOWS, 1000 * s + i)
    for i, (s, te) in enumerate(jobs))
E1_SC = defaultdict(lambda: np.full((len(cfg.e1_seeds), len(_yf)), np.nan))
for (s, te), out in zip(jobs, _res):
    r = list(cfg.e1_seeds).index(s)
    for key, sc in out.items():
        E1_SC[key][r, te] = sc
rows = []
for (wi, shape_only), S in E1_SC.items():
    kind, a, b, idx = WINDOWS[wi]
    rs = rec_scores(np.nanmean(S, 0), _g, E1_RECS)
    per_rep = [auc_safe(E1_Y, rec_scores(S[r], _g, E1_RECS)) for r in range(S.shape[0])]
    auc = auc_safe(E1_Y, rs); lo, hi = boot_auc_ci(E1_Y, rs, n_boot=nb_, seed=wi)
    rows.append(dict(kind=kind, variant="shape" if shape_only else "shape+level", f_lo=a, f_hi=b,
                     fc=np.sqrt(a * b), n_bands=len(idx), auc=auc, auc_lo=lo, auc_hi=hi, auc_rep_sd=np.nanstd(per_rep)))
E1D = pd.DataFrame(rows).sort_values(["kind", "variant", "f_lo", "f_hi"]).reset_index(drop=True)
savetab(E1D.round(4), "T05_band_sweep")
print(f"Quét {len(WINDOWS)} cửa sổ x {len(jobs)} fold trong {time.time() - t0:.0f} s")

# ---- Dải thiết yếu ------------------------------------------------------------
lp = E1D[(E1D.kind == "lowpass")].sort_values("f_hi"); hp = E1D[(E1D.kind == "highpass")].sort_values("f_lo")
AUC_FULL = float(lp.auc.iloc[-1])
fc_lp = float(lp.loc[lp.auc >= AUC_FULL - cfg.e1_tol, "f_hi"].min())
fc_hp = float(hp.loc[hp.auc >= AUC_FULL - cfg.e1_tol, "f_lo"].max())
oct_ = E1D[(E1D.kind == "octave") & (E1D.variant == "shape+level")].sort_values("auc", ascending=False)
ESS = None
if fc_hp < fc_lp and len(_win_idx(fc_hp, fc_lp)) >= 2:
    idx = _win_idx(fc_hp, fc_lp)
    _r2 = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(e1_fold_job)(E1_LB[~te], _yf[~te], _g[~te], E1_LB[te], [("ess", fc_hp, fc_lp, idx)], 1000 * s + i)
        for i, (s, te) in enumerate(jobs))
    S = np.full((len(cfg.e1_seeds), len(_yf)), np.nan)
    for (s, te), out in zip(jobs, _r2):
        S[list(cfg.e1_seeds).index(s), te] = out[(0, False)]
    rs = rec_scores(np.nanmean(S, 0), _g, E1_RECS)
    ESS = dict(f_lo=fc_hp, f_hi=fc_lp, auc=auc_safe(E1_Y, rs), ci=boot_auc_ci(E1_Y, rs, n_boot=nb_, seed=7))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for v, col in (("shape+level", CAT[0]), ("shape", CAT[2])):
    d = E1D[(E1D.kind == "octave") & (E1D.variant == v)].sort_values("fc")
    ax[0].fill_between(d.fc, d.auc_lo, d.auc_hi, color=col, alpha=0.15, lw=0)
    ax[0].plot(d.fc, d.auc, "-o", color=col, ms=3.5, label=f"octave window, {v}")
ax[0].axhline(0.5, color=INK2, lw=0.8); ax[0].axhline(AUC_FULL, color=CAT[1], lw=1.0)
ax[0].text(E1D.fc.min(), AUC_FULL + 0.01, f"full band {AUC_FULL:.3f}", color=INK2, fontsize=8)
freq_axis(ax[0], label="window centre frequency (Hz)"); ax[0].set_ylabel("record-level AUC")
ax[0].set_title("(a) Model AUC using one octave window only"); ax[0].legend(loc="lower right")
for d, col, lab, xcol in ((lp, CAT[0], "low-pass [f_lo, fc]", "f_hi"), (hp, CAT[1], "high-pass [fc, f_hi]", "f_lo")):
    ax[1].fill_between(d[xcol], d.auc_lo, d.auc_hi, color=col, alpha=0.15, lw=0)
    ax[1].plot(d[xcol], d.auc, "-o", color=col, ms=3.5, label=lab)
ax[1].axhline(AUC_FULL - cfg.e1_tol, color=INK2, lw=0.8)
ax[1].axvline(fc_lp, color=CAT[0], lw=0.8); ax[1].axvline(fc_hp, color=CAT[1], lw=0.8)
freq_axis(ax[1], label="cut-off frequency fc (Hz)"); ax[1].set_ylabel("record-level AUC")
ax[1].set_title(f"(b) Cumulative band sweep; essential band ≈ {fc_hp:.0f}-{fc_lp:.0f} Hz"); ax[1].legend(loc="lower center")
fig.tight_layout()
savefig(fig, "Fig06_band_sweep", "Band-limited model performance (octave windows, low-pass and high-pass sweeps).")

In [ ]:
# =========================================================
# 5.4 Kết luận RQ1 (sinh tự động từ số đo)
# =========================================================
_sig = E1B[E1B.q_bh < 0.05]
_hi_leak = E1B[(E1B.auc_lo > 0.5)]; _hi_nol = E1B[(E1B.auc_hi < 0.5)]


def _ranges(df):
    if not len(df):
        return "không có"
    df = df.sort_values("f_lo"); out = []; a, b = df.f_lo.iloc[0], df.f_hi.iloc[0]
    for lo_, hi_ in zip(df.f_lo.iloc[1:], df.f_hi.iloc[1:]):
        if lo_ <= b * 1.001:
            b = hi_
        else:
            out.append((a, b)); a, b = lo_, hi_
    out.append((a, b))
    return ", ".join(f"{x:.0f}-{y:.0f} Hz" for x, y in out)


E1_SUMMARY = dict(cond=E1_COND, n_rec=int(len(E1_RECS)), auc_full=AUC_FULL, fc_lp=fc_lp, fc_hp=fc_hp,
                  best_octave=f"{oct_.f_lo.iloc[0]:.0f}-{oct_.f_hi.iloc[0]:.0f} Hz", best_octave_auc=float(oct_.auc.iloc[0]),
                  leak_higher=_ranges(_hi_leak), noleak_higher=_ranges(_hi_nol), n_sig_fdr=int(len(_sig)), ess=ESS)
txt = [f"**RQ1 — Dải tần mang thông tin rò rỉ** ({E1_COND}, {len(E1_RECS)} bản ghi):",
       f"- Năng lượng tương đối **cao hơn ở bản ghi leak** (CI 95% của AUC nằm trên 0,5): {E1_SUMMARY['leak_higher']}.",
       f"- Năng lượng tương đối **cao hơn ở bản ghi noleak**: {E1_SUMMARY['noleak_higher']}.",
       f"- Số dải 1/6 octave còn ý nghĩa sau hiệu chỉnh FDR: {len(_sig)}/{len(E1B)}.",
       f"- Cửa sổ một octave tốt nhất: {E1_SUMMARY['best_octave']} (AUC {E1_SUMMARY['best_octave_auc']:.3f}); toàn dải: AUC {AUC_FULL:.3f}.",
       f"- Thông thấp: chỉ cần [{cfg.f_lo:.0f}, {fc_lp:.0f}] Hz để đạt AUC ≥ toàn dải − {cfg.e1_tol}; "
       f"thông cao: bỏ được phần dưới {fc_hp:.0f} Hz mà vẫn đạt ngưỡng đó."]
if ESS:
    txt.append(f"- **Dải thiết yếu {ESS['f_lo']:.0f}–{ESS['f_hi']:.0f} Hz**: AUC {fmt_ci(ESS['auc'], *ESS['ci'])} "
               f"(so với toàn dải {AUC_FULL:.3f}).")
else:
    txt.append(f"- fc_HP* ({fc_hp:.0f} Hz) ≥ fc_LP* ({fc_lp:.0f} Hz): thông tin phân biệt **lặp lại** ở cả dải thấp và dải cao "
               "(mỗi phía tự nó đã đủ) → không tồn tại một dải hẹp duy nhất; báo cáo cả hai phía.")
display(Markdown("\n".join(txt)))

## 6. Lõi học máy dùng chung / Shared machine-learning core

**Bốn mô hình** (được dùng nhiều nhất trong các công trình nhận dạng rò rỉ bằng âm thanh/rung động —
xem tổng quan của Fan, Tariq & Zayed 2022):

| Mô hình | Lý do chọn | Cài đặt |
|---|---|---|
| **SVM (RBF)** | chuẩn mực phổ biến nhất cho đặc trưng thủ công (Kang et al. 2018; Xu et al. 2021; Fares et al. 2023) | `SVC`, γ = hệ số × 1/k, trọng số mẫu cân bằng |
| **Random Forest** | bền với đặc trưng dư thừa, cho độ quan trọng (Breiman 2001; Ning et al. 2021) | 300 cây, trọng số mẫu cân bằng |
| **XGBoost** | gradient boosting mạnh nhất trên dữ liệu bảng (Chen & Guestrin 2016; Tijani et al. 2022) | 300 cây `hist` (thay bằng HistGradientBoosting nếu thiếu) |
| **k-NN** | mốc phi tham số kinh điển (Cover & Hart 1967; El-Zahab et al. 2018) | khoảng cách Euclid trên dữ liệu chuẩn hoá; hiệu chỉnh tiên nghiệm lớp |

**Giao thức chống rò rỉ dữ liệu** (Kaufman et al. 2012; Roberts et al. 2017; Varma & Simon 2006):
điền khuyết → chuẩn hoá → **chọn đặc trưng** → huấn luyện đều được `fit` **chỉ trên phần huấn luyện của
từng fold**; fold được chia trên **bảng bản ghi** (phân tầng theo lớp) nên không bản ghi nào xuất hiện ở
cả hai phía. Đầu ra mức khung được gộp về **mức bản ghi bằng trung bình xác suất** (quy tắc đăng ký
trước); ngưỡng quyết định 0,5 trên mô hình đã cân bằng lớp.

In [ ]:
# =========================================================
# 6.1 Tiền xử lý, mô hình, bộ xếp hạng đặc trưng
# =========================================================
CLIP_Z = 8.0
SUPPORTS_SW = {"SVM": True, "RF": True, "XGB": True, "KNN": False, "LR": True, "MLP": False}


class Prep:
    """Điền khuyết bằng trung vị + chuẩn hoá z + kẹp |z| <= 8 (fit trên TRAIN)."""

    def fit(self, X):
        X = np.asarray(X, np.float64)
        self.med = np.nanmedian(X, 0); self.med[~np.isfinite(self.med)] = 0.0
        Xf = np.where(np.isfinite(X), X, self.med)
        self.mu = Xf.mean(0); self.sd = Xf.std(0); self.sd[self.sd < 1e-12] = 1.0
        return self

    def transform(self, X):
        X = np.asarray(X, np.float64); X = np.where(np.isfinite(X), X, self.med)
        return np.clip((X - self.mu) / self.sd, -CLIP_Z, CLIP_Z).astype(np.float32)


def build_model(name, params=None, seed=0, n_feat=10, n_train=1000):
    p = dict(DEFAULT_PARAMS[name]); p.update(params or {})
    if name == "SVM":
        return SVC(kernel="rbf", C=p["C"], gamma=p["gamma_mult"] / max(n_feat, 1), cache_size=500)
    if name == "RF":
        return RandomForestClassifier(n_estimators=p["n_estimators"], max_features=p["max_features"],
                                      min_samples_leaf=p["min_samples_leaf"], n_jobs=1, random_state=seed)
    if name == "XGB":
        if HAS_XGB:
            return xgb.XGBClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"], learning_rate=p["learning_rate"],
                                     subsample=p["subsample"], colsample_bytree=p["colsample_bytree"],
                                     min_child_weight=p["min_child_weight"], reg_lambda=p["reg_lambda"],
                                     tree_method="hist", n_jobs=1, random_state=seed, verbosity=0, eval_metric="logloss")
        return HistGradientBoostingClassifier(max_iter=p["n_estimators"], max_depth=p["max_depth"],
                                              learning_rate=p["learning_rate"], random_state=seed)
    if name == "KNN":
        return KNeighborsClassifier(n_neighbors=int(min(p["n_neighbors"], max(1, n_train - 1))), weights=p["weights"], n_jobs=1)
    if name == "LR":
        return LogisticRegression(C=p["C"], max_iter=3000)
    if name == "MLP":
        return MLPClassifier(hidden_layer_sizes=p["hidden_layer_sizes"], alpha=p["alpha"], early_stopping=True,
                             max_iter=500, random_state=seed)
    raise ValueError(name)


def prior_correct(p, pi):
    """Đưa xác suất của mô hình KHÔNG cân bằng lớp về tiên nghiệm 50/50 (ngưỡng 0,5 = cân bằng)."""
    p = np.clip(p, 1e-6, 1 - 1e-6); a = p * 0.5 / pi; b = (1 - p) * 0.5 / (1 - pi)
    return a / (a + b)


def fit_model(name, A, y, g, params=None, seed=0):
    m = build_model(name, params, seed, n_feat=A.shape[1], n_train=len(y))
    if SUPPORTS_SW[name]:
        m.fit(A, y, sample_weight=sample_weights(y, g))
    else:
        m.fit(A, y)
    m.prior_ = float(np.mean(y))
    return m


def score_model(name, m, B):
    p = expit(m.decision_function(B)) if name == "SVM" else m.predict_proba(B)[:, 1]
    return prior_correct(p, m.prior_) if not SUPPORTS_SW[name] else p


# ---- Bộ xếp hạng đặc trưng (trả về thứ tự chỉ số, tốt nhất trước) ---------------
def _sub(A, y, g, n, seed):
    if len(y) <= n:
        return A, y, g
    i = np.random.default_rng(seed).choice(len(y), n, replace=False)
    return A[i], y[i], g[i]


def rank_anova(A, y, g, seed):
    F = np.nan_to_num(f_classif(A, y)[0]); return np.argsort(-F, kind="stable")


def rank_mi(A, y, g, seed):
    A2, y2, _ = _sub(A, y, g, min(cfg.rank_max_rows, 4000), seed)
    mi = mutual_info_classif(A2, y2, n_neighbors=3, random_state=seed)
    return np.lexsort((-np.nan_to_num(f_classif(A, y)[0]), -mi))


def rank_mrmr(A, y, g, seed, k_max=None):
    """mRMR dạng FCQ: độ liên quan = F-ANOVA, độ dư thừa = |tương quan Pearson| (Ding & Peng 2005; Peng et al. 2005)."""
    F = np.nan_to_num(f_classif(A, y)[0]); p = A.shape[1]; k_max = k_max or p
    C = np.nan_to_num(np.abs(np.corrcoef(A, rowvar=False)))
    sel = [int(np.argmax(F))]; red = C[:, sel[0]].copy(); used = np.zeros(p, bool); used[sel[0]] = True
    while len(sel) < min(k_max, p):
        sc = F / (red / len(sel) + 1e-3); sc[used] = -np.inf
        j = int(np.argmax(sc)); sel.append(j); used[j] = True; red += C[:, j]
    rest = [j for j in np.argsort(-F) if not used[j]]
    return np.array(sel + rest)


def rank_et(A, y, g, seed):
    A2, y2, g2 = _sub(A, y, g, cfg.rank_max_rows, seed)
    et = ExtraTreesClassifier(n_estimators=300, max_features="sqrt", min_samples_leaf=3, n_jobs=1, random_state=seed)
    et.fit(A2, y2, sample_weight=sample_weights(y2, g2))
    return np.argsort(-et.feature_importances_, kind="stable")


def rank_l1(A, y, g, seed):
    lr = LogisticRegression(penalty="l1", solver="liblinear", C=cfg.l1_C, max_iter=2000)
    lr.fit(A, y, sample_weight=sample_weights(y, g))
    return np.lexsort((-np.nan_to_num(f_classif(A, y)[0]), -np.abs(lr.coef_[0])))


RANKERS = {"anova": rank_anova, "mi": rank_mi, "mrmr": rank_mrmr, "et": rank_et, "l1": rank_l1}


def consensus(orders):
    """Xếp hạng đồng thuận Borda: trung bình vị trí qua các phương pháp (Saeys et al. 2008)."""
    p = len(orders[0]); pos = np.zeros((len(orders), p))
    for i, o in enumerate(orders):
        pos[i, o] = np.arange(p)
    return np.argsort(pos.mean(0), kind="stable")

In [ ]:
# =========================================================
# 6.2 Bộ máy CV lặp (song song theo fold) + tổng hợp chỉ số + so sánh ghép cặp
# =========================================================
class DS:
    """Khung nhìn dữ liệu của một điều kiện trên một phạm vi bản ghi."""

    def __init__(self, cond, scope, feats=None):
        df = FEAT[cond]; recs = np.sort(SCOPE_RECS[scope])
        m = df.record_id.isin(recs).values
        self.df = df.loc[m, META_COLS].reset_index(drop=True)
        self.feats = list(feats or FEATURES)
        self.X = np.array(df.loc[m, self.feats].to_numpy(np.float32), copy=True)
        self.y = self.df.label.to_numpy(int); self.g = self.df.record_id.to_numpy()
        self.recs = recs; self.rec_y = REC_LABEL.loc[recs].values
        self.cond, self.scope = cond, scope


def fold_job(Xtr, ytr, gtr, Xte, combos, rankers, seed):
    prep = Prep().fit(Xtr); A, B = prep.transform(Xtr), prep.transform(Xte)
    orders = {}
    need = [r for r in rankers if r != "consensus"]
    if "consensus" in rankers:
        need += [r for r in cfg.rankers if r not in need]
    for r in need:
        orders[r] = RANKERS[r](A, ytr, gtr, seed)
    if "consensus" in rankers:
        orders["consensus"] = consensus([orders[r] for r in cfg.rankers])
    out = {}
    for rk, k, model in combos:
        idx = np.arange(A.shape[1]) if (not k or rk is None) else orders[rk][:k]
        m = fit_model(model, A[:, idx], ytr, gtr, seed=seed)
        out[(rk, k, model)] = score_model(model, m, B[:, idx]).astype(np.float32)
    return out, {r: o.astype(np.int16) for r, o in orders.items()}


class CVRun:
    def __init__(self, ds, seeds):
        self.ds, self.seeds = ds, list(seeds)
        self.scores = {}; self.orders = defaultdict(list); self.fold_of = np.full((len(seeds), len(ds.y)), -1)


def run_cv(ds, combos, rankers=(), seeds=None, tag=""):
    seeds = list(seeds or cfg.seeds); t0 = time.time()
    run = CVRun(ds, seeds); jobs = []
    for r, s in enumerate(seeds):
        for fi, te_r in enumerate(FOLDS[(ds.scope, s)]):
            te = np.isin(ds.g, te_r); run.fold_of[r, te] = fi; jobs.append((r, fi, te))
    res = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(fold_job)(ds.X[~te], ds.y[~te], ds.g[~te], ds.X[te], combos, tuple(rankers), 7919 * seeds[r] + fi)
        for r, fi, te in jobs)
    for (r, fi, te), (out, orders) in zip(jobs, res):
        for key, sc in out.items():
            run.scores.setdefault(key, np.full((len(seeds), len(ds.y)), np.nan, np.float32))[r, te] = sc
        for rk, o in orders.items():
            run.orders[rk].append(o)
    print(f"  [{tag}] {ds.cond}/{ds.scope}: {len(jobs)} fold x {len(combos)} cấu hình trong {time.time() - t0:.0f} s")
    return run


def summarize(run, key, boot=True):
    ds = run.ds; S = run.scores[key]; R = S.shape[0]
    rep_rec = [rec_scores(S[r], ds.g, ds.recs) for r in range(R)]
    rec_auc_rep = [auc_safe(ds.rec_y, v) for v in rep_rec]
    fold_auc = []
    for r in range(R):
        for fi, te_r in enumerate(FOLDS[(ds.scope, run.seeds[r])]):
            ii = np.isin(ds.recs, te_r); fold_auc.append(auc_safe(ds.rec_y[ii], rep_rec[r][ii]))
    s_mean = np.nanmean(S, 0); rs = rec_scores(s_mean, ds.g, ds.recs)
    out = dict(rec_auc=auc_safe(ds.rec_y, rs), rec_auc_rep_mean=np.nanmean(rec_auc_rep), rec_auc_rep_sd=np.nanstd(rec_auc_rep),
               frame_auc=auc_safe(ds.y, s_mean), rec_ap=average_precision_score(ds.rec_y, rs))
    if boot:
        out["rec_auc_lo"], out["rec_auc_hi"] = boot_auc_ci(ds.rec_y, rs, seed=1)
        out["frame_auc_lo"], out["frame_auc_hi"] = boot_auc_ci(ds.y, s_mean, n_boot=max(200, cfg.n_boot // 2), seed=2, groups=ds.g)
    out.update({f"rec_{k}": v for k, v in bin_metrics(ds.rec_y, rs).items()
                if k in ("acc", "bacc", "sens", "spec", "f1", "mcc")})
    out["_rec_scores"] = rs; out["_fold_auc"] = np.asarray(fold_auc)
    return out


def compare(sumA, sumB, rec_y, n_splits=None):
    """So sánh ghép cặp A - B trên cùng các bản ghi: bootstrap, DeLong, Nadeau-Bengio."""
    d, p_dl = delong_test(rec_y, sumA["_rec_scores"], sumB["_rec_scores"])
    lo, hi, p_bt = paired_boot_delta(rec_y, sumA["_rec_scores"], sumB["_rec_scores"], seed=3)
    K = n_splits or cfg.n_splits
    p_nb = nb_ttest(sumA["_fold_auc"] - sumB["_fold_auc"], K - 1, 1)
    return dict(d_auc=d, d_lo=lo, d_hi=hi, p_delong=p_dl, p_boot=p_bt, p_nb=p_nb)


print("Lõi ML sẵn sàng | mô hình:", list(cfg.models), "| bộ xếp hạng:", list(cfg.rankers) + ["consensus"])

## 7. E2 — Xử lý tín hiệu có giúp nhận dạng tốt hơn tín hiệu thô? / Processed vs raw (RQ2)

**Thiết kế có kiểm soát**: chỉ thay đổi điều kiện tín hiệu (C0…C6); mọi thứ khác giữ nguyên — cùng
danh mục đặc trưng, cùng các fold mức bản ghi (ghép cặp), cùng 4 mô hình với **siêu tham số mặc định
cố định**, hai chế độ đặc trưng: `all` (toàn bộ) và `mrmr30` (30 đặc trưng mRMR chọn lại **trong từng
fold**). Chỉ dùng tập **DEV**.

Thống kê: AUC mức bản ghi (trung bình qua các lần lặp CV và AUC của điểm OOF trung bình kèm CI
bootstrap), so sánh **ghép cặp** với `C0_RAW` bằng DeLong (p hiệu chỉnh Holm theo từng mô hình),
CI bootstrap của ΔAUC và t-test hiệu chỉnh Nadeau–Bengio trên AUC từng fold.

**Quy tắc chọn điều kiện tốt nhất (đăng ký trước)**: AUC mức bản ghi trung bình qua 4 mô hình ở chế
độ `cfg.e2_primary`. Điều kiện này được dùng cho E3–E4.

In [ ]:
# =========================================================
# 7.1 Chạy E2: 7 điều kiện x 2 chế độ đặc trưng x 4 mô hình x (lặp x fold)
# =========================================================
def parse_regime(rg):
    if rg == "all":
        return None, 0
    m = re.fullmatch(r"([a-z]+)(\d+)", rg)
    return m.group(1), int(m.group(2))


E2_RUNS, E2_ROWS = {}, []
for cond in cfg.conditions:
    ds = DS(cond, "dev")
    combos, rks = [], set()
    for rg in cfg.e2_regimes:
        rk, k = parse_regime(rg)
        if rk:
            rks.add(rk)
        combos += [(rk, k, m) for m in cfg.models]
    run = run_cv(ds, combos, rankers=tuple(sorted(rks)), tag="E2")
    E2_RUNS[cond] = run
    for rg in cfg.e2_regimes:
        rk, k = parse_regime(rg)
        for m in cfg.models:
            s = summarize(run, (rk, k, m))
            E2_ROWS.append(dict(condition=cond, regime=rg, model=m, n_frames=len(ds.y),
                                **{a: b for a, b in s.items() if not a.startswith("_")}, _s=s))
E2_TAB = pd.DataFrame(E2_ROWS)
_show = ["condition", "regime", "model", "n_frames", "rec_auc_rep_mean", "rec_auc_rep_sd", "rec_auc", "rec_auc_lo",
         "rec_auc_hi", "frame_auc", "rec_bacc", "rec_sens", "rec_spec", "rec_f1", "rec_mcc"]
savetab(E2_TAB[_show].round(4), "T06_E2_processing_results")
PIV = {rg: E2_TAB[E2_TAB.regime == rg].pivot(index="model", columns="condition", values="rec_auc_rep_mean")
       .reindex(index=list(cfg.models), columns=list(cfg.conditions)) for rg in cfg.e2_regimes}
for rg in cfg.e2_regimes:
    display(Markdown(f"**Bảng 6{'ab'[list(cfg.e2_regimes).index(rg)]}. AUC mức bản ghi (trung bình {len(cfg.seeds)} lần lặp CV) — chế độ `{rg}`.**"))
    _t = PIV[rg].copy(); _t.loc["mean over models"] = _t.mean(0)
    display(_t.round(4))

MEAN_BY_COND = PIV[cfg.e2_primary].mean(0).sort_values(ascending=False)
BEST_COND = MEAN_BY_COND.index[0]
print("\nAUC trung bình 4 mô hình theo điều kiện (chế độ chính):")
print(MEAN_BY_COND.round(4).to_string())
print(f"=> Điều kiện tốt nhất (quy tắc đăng ký trước): {BEST_COND}")

In [ ]:
# =========================================================
# 7.2 So sánh ghép cặp: mỗi điều kiện vs C0_RAW, và từng bước so với bước liền trước
# =========================================================
STEP_PREV = {"C1_BPF": "C0_RAW", "C2_TR": "C1_BPF", "C3_IR": "C2_TR", "C4_MBG": "C3_IR", "C5_SS": "C3_IR", "C6_WT": "C3_IR"}
_get = {(c, rg, m): s for c, rg, m, s in zip(E2_TAB.condition, E2_TAB.regime, E2_TAB.model, E2_TAB["_s"])}
rec_y_dev = DS(cfg.conditions[0], "dev").rec_y
CMP = []
for rg in cfg.e2_regimes:
    for m in cfg.models:
        for kind, pairs in (("vs_raw", [(c, "C0_RAW") for c in cfg.conditions if c != "C0_RAW"]),
                            ("stepwise", [(c, STEP_PREV[c]) for c in cfg.conditions if c in STEP_PREV and STEP_PREV[c] in cfg.conditions])):
            rows = [dict(regime=rg, model=m, comparison=kind, A=a, B=b,
                         **compare(_get[(a, rg, m)], _get[(b, rg, m)], rec_y_dev)) for a, b in pairs]
            ph = holm([r["p_delong"] for r in rows])
            for r, p in zip(rows, ph):
                r["p_delong_holm"] = p
            CMP += rows
E2_CMP = pd.DataFrame(CMP)
savetab(E2_CMP.round(4), "T07_E2_paired_tests")
_v = E2_CMP[(E2_CMP.comparison == "vs_raw") & (E2_CMP.regime == cfg.e2_primary)]
display(Markdown(f"**Bảng 7. ΔAUC mức bản ghi so với C0_RAW (chế độ `{cfg.e2_primary}`), DeLong + Holm.**"))
display(_v.pivot(index="A", columns="model", values="d_auc").round(4))
display(_v.pivot(index="A", columns="model", values="p_delong_holm").round(4))

In [ ]:
# =========================================================
# 7.3 Hình 7-8: bản đồ nhiệt AUC và biểu đồ rừng ΔAUC so với tín hiệu thô
# =========================================================
fig, ax = plt.subplots(1, len(cfg.e2_regimes), figsize=(6.2 * len(cfg.e2_regimes), 2.9), squeeze=False)
vmin = np.nanmin([PIV[r].values.min() for r in cfg.e2_regimes]); vmax = np.nanmax([PIV[r].values.max() for r in cfg.e2_regimes])
for j, rg in enumerate(cfg.e2_regimes):
    M = PIV[rg].values; a = ax[0, j]
    im = a.imshow(M, cmap=SEQ, vmin=vmin, vmax=vmax, aspect="auto")
    for (i, k), v in np.ndenumerate(M):
        a.text(k, i, f"{v:.3f}", ha="center", va="center", fontsize=7.5,
               color="white" if (v - vmin) / max(vmax - vmin, 1e-9) > 0.6 else INK)
    a.set_xticks(range(M.shape[1])); a.set_xticklabels([c.replace("_", "\n") for c in PIV[rg].columns], fontsize=7.5)
    a.set_yticks(range(M.shape[0])); a.set_yticklabels(PIV[rg].index); a.grid(False)
    a.set_title(f"({'ab'[j]}) record-level AUC, features: {rg}")
fig.colorbar(im, ax=ax.ravel().tolist(), shrink=0.9, label="AUC")
savefig(fig, "Fig07_E2_heatmap", "Record-level AUC per processing condition and model.")

fig, ax = plt.subplots(figsize=(8.5, 3.6))
conds = [c for c in cfg.conditions if c != "C0_RAW"]
for i, m in enumerate(cfg.models):
    d = _v[_v.model == m].set_index("A").reindex(conds)
    y = np.arange(len(conds)) + (i - (len(cfg.models) - 1) / 2) * 0.17
    ax.errorbar(d.d_auc, y, xerr=[d.d_auc - d.d_lo, d.d_hi - d.d_auc], fmt="o", ms=4, color=MODEL_COLOR[m],
                ecolor=MODEL_COLOR[m], elinewidth=1.2, capsize=0, label=m)
ax.axvline(0, color=INK2, lw=0.8)
ax.set_yticks(range(len(conds))); ax.set_yticklabels(conds); ax.invert_yaxis()
ax.set_xlabel("ΔAUC vs C0_RAW (record level, 95% paired bootstrap CI)")
ax.set_title(f"Effect of each processing condition relative to raw audio ({cfg.e2_primary})")
ax.legend(ncol=len(cfg.models), loc="lower right")
fig.tight_layout()
savefig(fig, "Fig08_E2_delta_vs_raw", "Paired ΔAUC of each processing condition relative to raw audio.")

In [ ]:
# =========================================================
# 7.4 Kết luận RQ2 (sinh tự động)
# =========================================================
_raw = PIV[cfg.e2_primary]["C0_RAW"]; _best = PIV[cfg.e2_primary][BEST_COND]
_sig_better = _v[(_v.A == BEST_COND) & (_v.p_delong_holm < 0.05) & (_v.d_auc > 0)].model.tolist()
_sig_worse = _v[(_v.p_delong_holm < 0.05) & (_v.d_auc < 0)][["A", "model"]].values.tolist()
E2_SUMMARY = dict(best=BEST_COND, raw_mean=float(_raw.mean()), best_mean=float(_best.mean()),
                  delta_mean=float(_best.mean() - _raw.mean()), sig_better=_sig_better, sig_worse=_sig_worse,
                  ranking=MEAN_BY_COND.round(4).to_dict())
lines = [f"**RQ2 — Xử lý vs thô** (DEV, {len(rec_y_dev)} bản ghi, chế độ `{cfg.e2_primary}`):",
         f"- AUC trung bình 4 mô hình: C0_RAW = {_raw.mean():.3f}; tốt nhất {BEST_COND} = {_best.mean():.3f} "
         f"(Δ = {_best.mean() - _raw.mean():+.3f}).",
         "- Xếp hạng điều kiện: " + " > ".join(f"{c} ({v:.3f})" for c, v in MEAN_BY_COND.items()) + ".",
         f"- {BEST_COND} tốt hơn C0_RAW **có ý nghĩa** (DeLong, Holm < 0,05) ở: {', '.join(_sig_better) if _sig_better else 'không mô hình nào'}.",
         f"- Các cặp (điều kiện, mô hình) **kém hơn có ý nghĩa** so với thô: {_sig_worse if _sig_worse else 'không có'}."]
for c in ("C5_SS", "C6_WT"):
    if c in PIV[cfg.e2_primary]:
        lines.append(f"- {c}: AUC trung bình {PIV[cfg.e2_primary][c].mean():.3f} "
                     f"({PIV[cfg.e2_primary][c].mean() - _raw.mean():+.3f} so với thô) — kiểm chứng giả thuyết "
                     "\"khử nhiễu dừng xoá mất tiếng rò rỉ\".")
display(Markdown("\n".join(lines)))

## 8. E3 — Tuyển chọn đặc trưng: ít nhất mà không giảm hiệu năng / Feature selection (RQ3)

Trên điều kiện tốt nhất của E2 (`BEST_COND`), tập DEV:

1. **E3a – Liên quan đơn biến** (mức bản ghi): AUC, Cliff's δ, Mann–Whitney + FDR cho từng đặc trưng.
2. **E3b – Dư thừa**: phân cụm phân cấp theo |ρ Spearman|; đếm số cụm ở |ρ| ≥ 0,9.
3. **E3c – Xếp hạng đồng thuận TRONG TỪNG FOLD** từ 5 phương pháp thuộc 3 họ — lọc (ANOVA-F,
   thông tin tương hỗ), lọc đa biến (mRMR-FCQ), nhúng (Extra-Trees, hồi quy logistic L1) — gộp Borda.
   Việc chọn **lặp lại trong mỗi fold** là bắt buộc: chọn trên toàn bộ dữ liệu rồi mới CV sẽ cho kết quả
   lạc quan giả tạo (Ambroise & McLachlan 2002).
4. **Đường cong AUC theo số đặc trưng k** cho 4 mô hình →
   `K_MIN` = k nhỏ nhất có AUC trung bình ≥ AUC tốt nhất − `cfg.k_tol` (**câu trả lời RQ3**);
   `K_MAIN` = k tốt nhất trong khoảng `cfg.k_main_range` = [20, 50] (bộ làm việc chính).
5. **E3d – So sánh phương pháp chọn** ở k ∈ `cfg.method_cmp_k`; **độ ổn định** Nogueira et al. (2018).
6. **Danh sách cuối**: xếp hạng đồng thuận trung bình qua mọi fold → top-`K_MAIN` và top-`K_MIN`.

In [ ]:
# =========================================================
# 8.1 E3a + E3b: liên quan đơn biến và cấu trúc dư thừa
# =========================================================
DS3 = DS(BEST_COND, "dev")
_rm = pd.DataFrame(DS3.X, columns=FEATURES).groupby(DS3.g).median().reindex(DS3.recs)
_p, _n = DS3.rec_y == 1, DS3.rec_y == 0
UNI = []
for n in FEATURES:
    v = _rm[n].values
    UNI.append(dict(feature=n, group=FEAT_META.loc[n, "group"], auc_rec=auc_safe(DS3.rec_y, v),
                    auc_frame=auc_safe(DS3.y, DS3.X[:, FEATURES.index(n)]), cliffs_delta=cliffs_delta(v[_p], v[_n]),
                    p=mwu_p(v[_p], v[_n]), leak_median=np.nanmedian(v[_p]), noleak_median=np.nanmedian(v[_n])))
UNI = pd.DataFrame(UNI); UNI["q_bh"] = bh_fdr(UNI.p.fillna(1).values)
UNI["sep"] = (UNI.auc_rec - 0.5).abs() * 2
UNI = UNI.sort_values("sep", ascending=False).reset_index(drop=True)
savetab(UNI.round(5), "T08_univariate_relevance")
display(Markdown(f"**Bảng 8. 20 đặc trưng liên quan đơn biến mạnh nhất (mức bản ghi, {BEST_COND}).** "
                 f"Có {int((UNI.q_bh < 0.05).sum())}/{len(UNI)} đặc trưng còn ý nghĩa sau FDR."))
display(UNI.head(20)[["feature", "group", "auc_rec", "auc_frame", "cliffs_delta", "p", "q_bh"]].round(4))

_Xs = pd.DataFrame(DS3.X, columns=FEATURES)
RHO = _Xs.rank().corr().abs().fillna(0).to_numpy(copy=True)
np.fill_diagonal(RHO, 1.0)
_Z = sch.linkage(squareform(1 - RHO, checks=False), method="average")
N_CLUST90 = len(np.unique(sch.fcluster(_Z, t=0.1, criterion="distance")))
print(f"Dư thừa: {len(FEATURES)} đặc trưng tạo thành {N_CLUST90} cụm ở |ρ| ≥ 0,9 "
      f"-> số chiều 'hiệu dụng' nhỏ hơn nhiều so với số đặc trưng.")

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1, 1.1]})
top = UNI.head(25).iloc[::-1]
ax[0].barh(range(len(top)), top.auc_rec - 0.5, left=0.5, color=[GROUP_COLOR[g] for g in top.group], height=0.7)
ax[0].axvline(0.5, color=INK2, lw=0.8)
ax[0].set_yticks(range(len(top))); ax[0].set_yticklabels(top.feature, fontsize=7)
ax[0].set_xlabel("record-level AUC (>0.5: higher in leak)"); ax[0].set_title("(a) Top-25 univariate features")
for g in GROUPS:
    if g in set(top.group):
        ax[0].barh([np.nan], [0], color=GROUP_COLOR[g], label=g)
ax[0].legend(fontsize=7, loc="lower right")
t40 = UNI.head(40).feature.tolist(); ii = [FEATURES.index(n) for n in t40]
sub = RHO[np.ix_(ii, ii)]; o = sch.leaves_list(sch.linkage(squareform(1 - sub, checks=False), "average"))
im = ax[1].imshow(sub[np.ix_(o, o)], cmap=SEQ, vmin=0, vmax=1); ax[1].grid(False)
ax[1].set_xticks(range(len(o))); ax[1].set_xticklabels([t40[k] for k in o], rotation=90, fontsize=5.5)
ax[1].set_yticks(range(len(o))); ax[1].set_yticklabels([t40[k] for k in o], fontsize=5.5)
ax[1].set_title("(b) |Spearman ρ| among the top-40 features (clustered)")
fig.colorbar(im, ax=ax[1], shrink=0.8)
fig.tight_layout()
savefig(fig, "Fig09_univariate_redundancy", "Univariate relevance and redundancy structure.")

In [ ]:
# =========================================================
# 8.2 E3c + E3d: xếp hạng TRONG fold, đường cong AUC(k), so sánh phương pháp
# =========================================================
_k = [k for k in cfg.k_grid if k == 0 or k < len(FEATURES)]
combos = [("consensus", k, m) for k in _k for m in cfg.models]
combos += [(r, k, m) for r in list(cfg.rankers) for k in cfg.method_cmp_k for m in cfg.method_cmp_models]
combos = list(dict.fromkeys(combos))
E3_RUN = run_cv(DS3, combos, rankers=tuple(cfg.rankers) + ("consensus",), tag="E3")

rows = []
for (rk, k, m) in combos:
    s = summarize(E3_RUN, (rk, k, m), boot=False)
    rows.append(dict(ranker=rk, k=(k if k else len(FEATURES)), model=m,
                     **{a: b for a, b in s.items() if not a.startswith("_")}))
E3_TAB = pd.DataFrame(rows)
CURVE = E3_TAB[E3_TAB.ranker == "consensus"].pivot(index="k", columns="model", values="rec_auc_rep_mean")[list(cfg.models)]
CURVE_F = E3_TAB[E3_TAB.ranker == "consensus"].pivot(index="k", columns="model", values="frame_auc")[list(cfg.models)]
CURVE["mean"] = CURVE.mean(1)
best_auc = CURVE["mean"].max()
K_MIN = int(CURVE.index[CURVE["mean"] >= best_auc - cfg.k_tol].min())
_rng = CURVE.loc[(CURVE.index >= cfg.k_main_range[0]) & (CURVE.index <= cfg.k_main_range[1])]
K_MAIN = int(_rng["mean"].idxmax()) if len(_rng) else int(CURVE["mean"].idxmax())
K_MIN_MODEL = {m: int(CURVE.index[CURVE[m] >= CURVE[m].max() - cfg.k_tol].min()) for m in cfg.models}
display(Markdown(f"**Bảng 9. AUC mức bản ghi theo số đặc trưng k (xếp hạng đồng thuận chọn trong fold, {BEST_COND}).**"))
display(CURVE.round(4))
savetab(E3_TAB.round(4), "T09_E3_k_curve_and_methods")
print(f"AUC trung bình tốt nhất = {best_auc:.4f} tại k = {int(CURVE['mean'].idxmax())}")
print(f"K_MIN  (ít nhất, giảm <= {cfg.k_tol}) = {K_MIN}  | theo từng mô hình: {K_MIN_MODEL}")
print(f"K_MAIN (tốt nhất trong {cfg.k_main_range}) = {K_MAIN}")

MCMP = E3_TAB[(E3_TAB.k.isin(cfg.method_cmp_k)) & (E3_TAB.model.isin(cfg.method_cmp_models))] \
    .pivot_table(index=["ranker"], columns=["model", "k"], values="rec_auc_rep_mean")
display(Markdown("**Bảng 10. So sánh phương pháp chọn đặc trưng (AUC mức bản ghi).**")); display(MCMP.round(4))
savetab(MCMP.round(4).reset_index(), "T10_E3_selection_methods")


def nogueira_stability(orders, k, p):
    Z = np.zeros((len(orders), p))
    for i, o in enumerate(orders):
        Z[i, o[:k]] = 1
    M = len(orders); pf = Z.mean(0); s2 = M / (M - 1) * pf * (1 - pf)
    kb = Z.sum(1).mean()
    return float(1 - s2.mean() / ((kb / p) * (1 - kb / p)))


STAB = pd.DataFrame([dict(ranker=r, k=k, stability=nogueira_stability(E3_RUN.orders[r], k, len(FEATURES)))
                     for r in list(cfg.rankers) + ["consensus"] for k in sorted(set(cfg.method_cmp_k) | {K_MIN, K_MAIN})])
display(Markdown("**Bảng 11. Độ ổn định lựa chọn qua các fold (Nogueira; 1 = luôn chọn cùng một tập).**"))
display(STAB.pivot(index="ranker", columns="k", values="stability").round(3))
savetab(STAB.round(4), "T11_selection_stability")

In [ ]:
# =========================================================
# 8.3 Danh sách đặc trưng cuối cùng (xếp hạng đồng thuận trung bình qua mọi fold DEV)
# =========================================================
_pos = np.zeros((len(E3_RUN.orders["consensus"]), len(FEATURES)))
for i, o in enumerate(E3_RUN.orders["consensus"]):
    _pos[i, o] = np.arange(len(FEATURES))
FINAL_RANK = np.argsort(_pos.mean(0), kind="stable")
FEAT_MAIN = [FEATURES[i] for i in FINAL_RANK[:K_MAIN]]
FEAT_MIN = [FEATURES[i] for i in FINAL_RANK[:K_MIN]]
sel_freq = lambda k: (_pos < k).mean(0)
SEL = pd.DataFrame(dict(rank=np.arange(1, K_MAIN + 1), feature=FEAT_MAIN))
SEL = SEL.merge(FEAT_META.reset_index()[["feature", "group", "family", "description", "level_dep", "band_lo", "band_hi"]], on="feature")
SEL["sel_freq_topKmain"] = [sel_freq(K_MAIN)[FEATURES.index(n)] for n in SEL.feature]
SEL["mean_rank"] = [_pos.mean(0)[FEATURES.index(n)] + 1 for n in SEL.feature]
SEL = SEL.merge(UNI[["feature", "auc_rec", "cliffs_delta", "q_bh"]], on="feature")
SEL["direction"] = np.where(SEL.auc_rec >= 0.5, "higher in leak", "higher in noleak")
SEL["in_K_MIN"] = SEL["rank"] <= K_MIN
display(Markdown(f"**Bảng 12. {K_MAIN} đặc trưng được chọn (K_MAIN); {K_MIN} dòng đầu = bộ tối thiểu K_MIN.**"))
display(SEL.round(3))
savetab(SEL.round(4), "T12_selected_features")
(OUT / "selected_features.json").write_text(json.dumps(
    {"condition": BEST_COND, "K_MAIN": FEAT_MAIN, "K_MIN": FEAT_MIN}, indent=1, ensure_ascii=False))

fig = plt.figure(figsize=(12.5, 4.2)); gs = gridspec.GridSpec(1, 3, width_ratios=[1.35, 1, 0.8], wspace=0.35)
a = fig.add_subplot(gs[0])
for m in cfg.models:
    a.plot(CURVE.index, CURVE[m], "-o", ms=3, color=MODEL_COLOR[m], lw=1.3, label=m)
a.plot(CURVE.index, CURVE["mean"], "-", color=INK, lw=2.0, label="mean of 4 models")
a.axhline(best_auc - cfg.k_tol, color=INK2, lw=0.8)
a.axvline(K_MIN, color=INK2, lw=0.8); a.axvline(K_MAIN, color=CAT[1], lw=0.8)
a.text(K_MIN, a.get_ylim()[0], f" K_MIN={K_MIN}", va="bottom", fontsize=8, color=INK2)
a.text(K_MAIN, a.get_ylim()[0] + 0.02, f" K_MAIN={K_MAIN}", va="bottom", fontsize=8, color=INK2)
a.set_xscale("log"); a.set_xlabel("number of selected features k (in-fold consensus ranking)")
a.set_ylabel("record-level AUC (DEV CV)"); a.set_title("(a) Performance vs number of features"); a.legend(fontsize=7)
b = fig.add_subplot(gs[1])
_sfr = SEL.iloc[::-1]
b.barh(range(len(_sfr)), _sfr.sel_freq_topKmain, color=[GROUP_COLOR[g] for g in _sfr.group], height=0.7)
b.set_yticks(range(len(_sfr))); b.set_yticklabels(_sfr.feature, fontsize=5.5 if K_MAIN > 30 else 7)
b.set_xlabel(f"selection frequency in top-{K_MAIN} across folds"); b.set_title("(b) Selected features")
c = fig.add_subplot(gs[2])
comp = pd.DataFrame({"K_MIN": SEL[SEL.in_K_MIN].group.value_counts(), "K_MAIN": SEL.group.value_counts()}).reindex(GROUPS).fillna(0)
left = np.zeros(2)
for g in GROUPS:
    v = comp.loc[g, ["K_MIN", "K_MAIN"]].values
    if v.sum():
        c.barh([0, 1], v, left=left, color=GROUP_COLOR[g], edgecolor="white", linewidth=1.5, label=g, height=0.6)
    left += v
c.set_yticks([0, 1]); c.set_yticklabels(["K_MIN", "K_MAIN"]); c.set_xlabel("features"); c.grid(False, axis="y")
c.set_title("(c) Composition by group"); c.legend(fontsize=6.5, loc="upper left", bbox_to_anchor=(1.02, 1.0), ncol=1)
savefig(fig, "Fig10_feature_selection", "AUC vs number of selected features, selected features and group composition.")

lines = [f"**RQ3 — Tuyển chọn đặc trưng** ({BEST_COND}, DEV):",
         f"- {len(FEATURES)} đặc trưng ban đầu chỉ tạo ~{N_CLUST90} cụm độc lập (|ρ| < 0,9) → dư thừa lớn.",
         f"- AUC trung bình 4 mô hình: toàn bộ = {CURVE.loc[len(FEATURES), 'mean']:.3f}; tốt nhất = {best_auc:.3f} "
         f"(k = {int(CURVE['mean'].idxmax())}).",
         f"- **Bộ tối thiểu K_MIN = {K_MIN} đặc trưng** giữ AUC trong phạm vi {cfg.k_tol} so với tốt nhất "
         f"(AUC {CURVE.loc[K_MIN, 'mean']:.3f}); theo từng mô hình: {K_MIN_MODEL}.",
         f"- **Bộ làm việc K_MAIN = {K_MAIN} đặc trưng** (AUC {CURVE.loc[K_MAIN, 'mean']:.3f}).",
         f"- Thành phần K_MIN theo nhóm: {SEL[SEL.in_K_MIN].group.value_counts().to_dict()}.",
         f"- Độ ổn định (đồng thuận, k = K_MAIN): {STAB[(STAB.ranker == 'consensus') & (STAB.k == K_MAIN)].stability.iloc[0]:.2f}."]
display(Markdown("\n".join(lines)))

## 9. E4 — Bốn mô hình cuối: CV lồng trên DEV + đánh giá một lần trên tập KIỂM TRA KHOÁ

- **CV lồng** (nested CV; Varma & Simon 2006; Cawley & Talbot 2010): vòng ngoài = CV lặp theo bản ghi
  (như E2–E3); trong mỗi fold ngoài: (i) xếp hạng đồng thuận **lại từ đầu** trên phần huấn luyện và lấy
  top-`K`; (ii) chọn siêu tham số bằng lưới + CV trong `cfg.inner_splits` fold **nhóm theo bản ghi**;
  (iii) huấn luyện lại và dự đoán fold ngoài. → ước lượng DEV không thiên lệch do lựa chọn.
- **Mô hình cuối**: huấn luyện trên toàn bộ DEV với danh sách đặc trưng cuối của E3 và siêu tham số chọn
  bằng CV nhóm trên DEV → đánh giá **đúng một lần** trên TEST.
- Hai bộ đặc trưng: `K_MAIN` (bộ làm việc 20–50) và `K_MIN` (bộ tối thiểu, `cfg.run_compact`).
- **E4c – khẳng định RQ2 trên TEST**: cùng 4 mô hình (mặc định, toàn bộ đặc trưng) huấn luyện trên DEV
  ở `C0_RAW` và ở `BEST_COND`, so sánh ghép cặp trên cùng các bản ghi TEST (DeLong).

In [ ]:
# =========================================================
# 9.1 CV lồng trên DEV
# =========================================================
def inner_folds(g, y, k, seed):
    recs = np.unique(g); ry = pd.Series(y).groupby(g).first().reindex(recs).values
    k = int(max(2, min(k, np.bincount(ry).min())))
    return [(~np.isin(g, recs[te]), np.isin(g, recs[te]))
            for _, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(recs, ry)]


def tune(model, A, y, g, inner, seed):
    best, best_s = None, -np.inf
    for params in ParameterGrid(PARAM_GRID[model]):
        sc = []
        for tr, va in inner:
            m = fit_model(model, A[tr], y[tr], g[tr], params, seed)
            sc.append(auc_safe(y[va], score_model(model, m, A[va])))
        if np.nanmean(sc) > best_s:
            best, best_s = params, float(np.nanmean(sc))
    return best, best_s


def nested_job(Xtr, ytr, gtr, Xte, K, models, seed):
    prep = Prep().fit(Xtr); A, B = prep.transform(Xtr), prep.transform(Xte)
    idx = consensus([RANKERS[r](A, ytr, gtr, seed) for r in cfg.rankers])[:K]
    A2, B2 = A[:, idx], B[:, idx]
    inner = inner_folds(gtr, ytr, cfg.inner_splits, seed); out = {}
    for m in models:
        best, bs = tune(m, A2, ytr, gtr, inner, seed)
        mdl = fit_model(m, A2, ytr, gtr, best, seed)
        out[m] = (score_model(m, mdl, B2).astype(np.float32), best, bs)
    return out, idx.astype(np.int16)


def run_nested(cond, K, tag):
    ds = DS(cond, "dev"); run = CVRun(ds, cfg.seeds); jobs = []; t0 = time.time()
    for r, s in enumerate(cfg.seeds):
        for fi, te_r in enumerate(FOLDS[("dev", s)]):
            te = np.isin(ds.g, te_r); run.fold_of[r, te] = fi; jobs.append((r, fi, te))
    res = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(nested_job)(ds.X[~te], ds.y[~te], ds.g[~te], ds.X[te], K, tuple(cfg.models), 104729 * cfg.seeds[r] + fi)
        for r, fi, te in jobs)
    params = defaultdict(list)
    for (r, fi, te), (out, idx) in zip(jobs, res):
        for m, (sc, best, bs) in out.items():
            run.scores.setdefault((None, K, m), np.full((len(cfg.seeds), len(ds.y)), np.nan, np.float32))[r, te] = sc
            params[m].append(json.dumps(best, default=str))
        run.orders["consensus"].append(idx)
    print(f"  [{tag}] CV lồng {cond}, K={K}: {len(jobs)} fold ngoài trong {time.time() - t0:.0f} s")
    return run, params


FEATSETS = {"K_MAIN": (K_MAIN, FEAT_MAIN)}
if cfg.run_compact and K_MIN != K_MAIN:
    FEATSETS["K_MIN"] = (K_MIN, FEAT_MIN)
NESTED, E4_ROWS = {}, []
for fs, (K, _) in FEATSETS.items():
    run, params = run_nested(BEST_COND, K, f"E4-{fs}")
    NESTED[fs] = run
    for m in cfg.models:
        s = summarize(run, (None, K, m))
        common = Counter(params[m]).most_common(1)[0]
        E4_ROWS.append(dict(featset=fs, k=K, model=m, **{a: b for a, b in s.items() if not a.startswith("_")},
                            modal_params=common[0], modal_freq=common[1] / len(params[m]), _s=s))
E4_DEV = pd.DataFrame(E4_ROWS)
_c = ["featset", "k", "model", "rec_auc_rep_mean", "rec_auc_rep_sd", "rec_auc", "rec_auc_lo", "rec_auc_hi", "frame_auc",
      "frame_auc_lo", "frame_auc_hi", "rec_ap", "rec_acc", "rec_bacc", "rec_sens", "rec_spec", "rec_f1", "rec_mcc", "modal_params"]
display(Markdown(f"**Bảng 13. Kết quả CV lồng trên DEV ({BEST_COND}).**")); display(E4_DEV[_c].round(4))
savetab(E4_DEV[_c].round(4), "T13_E4_nested_dev")

_pw = []
for fs in FEATSETS:
    sub = E4_DEV[E4_DEV.featset == fs].set_index("model")
    pairs = list(itertools.combinations(cfg.models, 2))
    rows = [dict(featset=fs, A=a, B=b, **compare(sub.at[a, "_s"], sub.at[b, "_s"], NESTED[fs].ds.rec_y)) for a, b in pairs]
    for r, p in zip(rows, holm([r["p_delong"] for r in rows])):
        r["p_delong_holm"] = p
    _pw += rows
E4_PAIR_DEV = pd.DataFrame(_pw); savetab(E4_PAIR_DEV.round(4), "T13b_E4_pairwise_dev")
BEST_MODEL = E4_DEV[E4_DEV.featset == "K_MAIN"].sort_values("rec_auc_rep_mean", ascending=False).model.iloc[0]
print(f"Mô hình tốt nhất trên DEV (K_MAIN): {BEST_MODEL}")
display(E4_PAIR_DEV[["featset", "A", "B", "d_auc", "d_lo", "d_hi", "p_delong_holm", "p_nb"]].round(4))

In [ ]:
# =========================================================
# 9.2 Mô hình cuối (huấn luyện trên toàn bộ DEV) -> đánh giá MỘT LẦN trên TEST
# =========================================================
def final_job(model, A, y, g, B, seed):
    best, bs = tune(model, A, y, g, inner_folds(g, y, cfg.inner_splits, seed), seed)
    mdl = fit_model(model, A, y, g, best, seed)
    return model, mdl, best, (score_model(model, mdl, B) if len(B) else np.array([]))


FINAL, TEST_ROWS, TEST_SC = {}, [], {}
for fs, (K, feats) in FEATSETS.items():
    dsd = DS(BEST_COND, "dev", feats); dst = DS(BEST_COND, "test", feats) if HAS_TEST else None
    prep = Prep().fit(dsd.X); A = prep.transform(dsd.X)
    B = prep.transform(dst.X) if HAS_TEST else np.zeros((0, len(feats)), np.float32)
    outs = Parallel(n_jobs=min(N_JOBS, len(cfg.models)), backend="loky")(
        delayed(final_job)(m, A, dsd.y, dsd.g, B, 2024) for m in cfg.models)
    for m, mdl, best, p in outs:
        FINAL[(fs, m)] = dict(prep=dict(med=prep.med, mu=prep.mu, sd=prep.sd, clip_z=CLIP_Z), model=mdl, features=feats, condition=BEST_COND, params=best, sr=SR,
                              frame_sec=cfg.frame_sec, prior=mdl.prior_, name=m)
        joblib.dump(FINAL[(fs, m)], MODEL_DIR / f"final_{m}_{fs}_k{K}.joblib", compress=3)
        if HAS_TEST:
            rs = rec_scores(p, dst.g, dst.recs); TEST_SC[(fs, m)] = (p, rs)
            lo, hi = boot_auc_ci(dst.rec_y, rs, seed=5)
            flo, fhi = boot_auc_ci(dst.y, p, groups=dst.g, n_boot=max(200, cfg.n_boot // 2), seed=6)
            TEST_ROWS.append(dict(featset=fs, k=K, model=m, n_rec=len(dst.recs), rec_auc=auc_safe(dst.rec_y, rs), rec_auc_lo=lo,
                                  rec_auc_hi=hi, frame_auc=auc_safe(dst.y, p), frame_auc_lo=flo, frame_auc_hi=fhi,
                                  rec_ap=average_precision_score(dst.rec_y, rs),
                                  **{f"rec_{a}": b for a, b in bin_metrics(dst.rec_y, rs).items()}, params=json.dumps(best, default=str)))
if HAS_TEST:
    DST = DS(BEST_COND, "test", FEAT_MAIN)
    E4_TEST = pd.DataFrame(TEST_ROWS)
    display(Markdown(f"**Bảng 14. Kết quả trên TẬP KIỂM TRA KHOÁ ({len(DST.recs)} bản ghi: "
                     f"{DST.rec_y.sum()} leak / {(1 - DST.rec_y).sum()} noleak).**"))
    display(E4_TEST.round(4)); savetab(E4_TEST.round(4), "T14_E4_test")
    _pt = []
    for fs in FEATSETS:
        pairs = list(itertools.combinations(cfg.models, 2))
        rows = []
        for a, b in pairs:
            d, p = delong_test(DST.rec_y, TEST_SC[(fs, a)][1], TEST_SC[(fs, b)][1])
            rows.append(dict(featset=fs, A=a, B=b, d_auc=d, p_delong=p))
        for r, p in zip(rows, holm([r["p_delong"] for r in rows])):
            r["p_delong_holm"] = p
        _pt += rows
    E4_PAIR_TEST = pd.DataFrame(_pt); savetab(E4_PAIR_TEST.round(4), "T14b_E4_pairwise_test")
else:
    print("Không có tập kiểm tra khoá (cfg.holdout_frac = 0): chỉ báo cáo CV lồng trên DEV.")

In [ ]:
# =========================================================
# 9.3 Hình 11-12: ROC/PR mức bản ghi (DEV CV lồng, TEST) và ma trận nhầm lẫn TEST
# =========================================================
def _roc_pr(axr, axp, y, scores_by_model, title):
    for m, s in scores_by_model.items():
        fpr, tpr, _ = roc_curve(y, s); pr, rc, _ = precision_recall_curve(y, s)
        axr.plot(fpr, tpr, color=MODEL_COLOR[m], lw=1.5, label=f"{m} ({auc_safe(y, s):.3f})")
        axp.plot(rc, pr, color=MODEL_COLOR[m], lw=1.5, label=f"{m} ({average_precision_score(y, s):.3f})")
    axr.plot([0, 1], [0, 1], color=GRID, lw=1); axp.axhline(np.mean(y), color=GRID, lw=1)
    axr.set_xlabel("false positive rate"); axr.set_ylabel("true positive rate"); axr.set_title(f"ROC - {title}")
    axp.set_xlabel("recall"); axp.set_ylabel("precision"); axp.set_title(f"PR - {title}")
    axr.legend(loc="lower right", title="AUC"); axp.legend(loc="lower left", title="AP")


ncol = 2 if HAS_TEST else 1
fig, ax = plt.subplots(2, ncol, figsize=(5.4 * ncol, 8.4), squeeze=False)
_dev_sc = {m: E4_DEV[(E4_DEV.featset == "K_MAIN") & (E4_DEV.model == m)]["_s"].iloc[0]["_rec_scores"] for m in cfg.models}
_roc_pr(ax[0, 0], ax[1, 0], NESTED["K_MAIN"].ds.rec_y, _dev_sc, f"DEV nested CV (k={K_MAIN})")
if HAS_TEST:
    _roc_pr(ax[0, 1], ax[1, 1], DST.rec_y, {m: TEST_SC[("K_MAIN", m)][1] for m in cfg.models}, f"TEST (k={K_MAIN})")
fig.tight_layout()
savefig(fig, "Fig11_roc_pr", "Record-level ROC and PR curves (DEV nested CV and hold-out TEST).")

if HAS_TEST:
    fig, ax = plt.subplots(1, len(cfg.models) + 1, figsize=(3.0 * (len(cfg.models) + 1), 3.0),
                           gridspec_kw={"width_ratios": [1] * len(cfg.models) + [1.6]})
    for j, m in enumerate(cfg.models):
        cm = confusion_matrix(DST.rec_y, (TEST_SC[("K_MAIN", m)][1] >= 0.5).astype(int), labels=[0, 1])
        ax[j].imshow(cm, cmap=SEQ, vmin=0, vmax=cm.sum()); ax[j].grid(False)
        for (i, k), v in np.ndenumerate(cm):
            ax[j].text(k, i, str(v), ha="center", va="center", color="white" if v > cm.sum() / 2 else INK, fontsize=11)
        ax[j].set_xticks([0, 1]); ax[j].set_xticklabels(["noleak", "leak"]); ax[j].set_yticks([0, 1])
        ax[j].set_yticklabels(["noleak", "leak"] if j == 0 else ["", ""]); ax[j].set_xlabel("predicted")
        ax[j].set_title(m)
    ax[0].set_ylabel("true")
    a = ax[-1]
    for j, m in enumerate(cfg.models):
        rs = TEST_SC[("K_MAIN", m)][1]
        for c, yv in (("noleak", 0), ("leak", 1)):
            v = rs[DST.rec_y == yv]
            a.scatter(np.full(len(v), j) + (yv - 0.5) * 0.3 + np.random.default_rng(j).uniform(-0.06, 0.06, len(v)), v,
                      s=16, color=PAL[c], edgecolor="white", linewidth=0.5, label=c if j == 0 else None)
    a.axhline(0.5, color=INK2, lw=0.8); a.set_xticks(range(len(cfg.models))); a.set_xticklabels(cfg.models)
    a.set_ylabel("recording score (mean frame probability)"); a.set_title("TEST recordings"); a.legend(fontsize=7)
    fig.tight_layout()
    savefig(fig, "Fig12_test_confusion_scores", "Hold-out TEST confusion matrices and per-recording scores.")

In [ ]:
# =========================================================
# 9.4 E4c: khẳng định RQ2 trên TEST - C0_RAW vs BEST_COND (toàn bộ đặc trưng, tham số mặc định)
# =========================================================
if HAS_TEST and BEST_COND != "C0_RAW":
    def raw_vs_best_job(cond, m, Xd, yd, gd, Xt, gt, recs_t):
        prep = Prep().fit(Xd)
        mdl = fit_model(m, prep.transform(Xd), yd, gd, seed=11)
        return cond, m, rec_scores(score_model(m, mdl, prep.transform(Xt)), gt, recs_t)
    _dd = {c: (DS(c, "dev"), DS(c, "test")) for c in ("C0_RAW", BEST_COND)}
    outs = Parallel(n_jobs=N_JOBS, backend="loky")(
        delayed(raw_vs_best_job)(c, m, _dd[c][0].X, _dd[c][0].y, _dd[c][0].g, _dd[c][1].X, _dd[c][1].g, _dd[c][1].recs)
        for c in ("C0_RAW", BEST_COND) for m in cfg.models)
    RS = {(c, m): v for c, m, v in outs}
    rows = []
    for m in cfg.models:
        d, p = delong_test(DST.rec_y, RS[(BEST_COND, m)], RS[("C0_RAW", m)])
        lo, hi, _ = paired_boot_delta(DST.rec_y, RS[(BEST_COND, m)], RS[("C0_RAW", m)], seed=9)
        rows.append(dict(model=m, auc_raw=auc_safe(DST.rec_y, RS[("C0_RAW", m)]), auc_best=auc_safe(DST.rec_y, RS[(BEST_COND, m)]),
                         d_auc=d, d_lo=lo, d_hi=hi, p_delong=p))
    E4C = pd.DataFrame(rows); E4C["p_delong_holm"] = holm(E4C.p_delong.values)
    display(Markdown(f"**Bảng 15. TEST: {BEST_COND} so với C0_RAW (toàn bộ đặc trưng, tham số mặc định).**"))
    display(E4C.round(4)); savetab(E4C.round(4), "T15_E4c_raw_vs_best_test")
else:
    E4C = None

## 10. E5 — Diễn giải, độ bền và tính ứng dụng / Interpretation, robustness and practicality (RQ4)

1. **Độ quan trọng đặc trưng** của mô hình tốt nhất: permutation importance trên TEST; SHAP (TreeExplainer;
   Lundberg et al. 2020) cho mô hình cây tốt nhất; **ánh xạ độ quan trọng lên trục tần số** (đối chiếu RQ1).
2. **Kiểm tra confound mức tín hiệu**: mô hình chỉ dùng mức RMS; mô hình bỏ mọi đặc trưng phụ thuộc mức.
3. **Minh hoạ rò rỉ dữ liệu**: chia fold ngẫu nhiên theo KHUNG (sai) so với theo BẢN GHI (đúng).
4. **Thời gian nghe tối thiểu**: AUC mức bản ghi theo số giây nghe liên tục (giá trị thực tiễn cho người vận hành).
5. **Chi phí tính toán** (trích đặc trưng + suy luận) và **phân tích lỗi** theo bản ghi.

In [ ]:
# =========================================================
# 10.1 Độ quan trọng đặc trưng: permutation (TEST) + SHAP + ánh xạ tần số
# =========================================================
_eval = DS(BEST_COND, "test" if HAS_TEST else "dev", FEAT_MAIN)
_fin = FINAL[("K_MAIN", BEST_MODEL)]
_P = _fin["prep"]
_B = np.clip((np.where(np.isfinite(_eval.X), _eval.X, _P["med"]) - _P["mu"]) / _P["sd"], -CLIP_Z, CLIP_Z).astype(np.float32)
base = auc_safe(_eval.y, score_model(BEST_MODEL, _fin["model"], _B))
rng = np.random.default_rng(0); n_rep = 5 if cfg.smoke_test else 20; imp = []
for j, n in enumerate(FEAT_MAIN):
    drops = []
    for _ in range(n_rep):
        Bp = _B.copy(); Bp[:, j] = rng.permutation(Bp[:, j])
        drops.append(base - auc_safe(_eval.y, score_model(BEST_MODEL, _fin["model"], Bp)))
    imp.append(dict(feature=n, perm_auc_drop=np.mean(drops), perm_sd=np.std(drops)))
PIMP = pd.DataFrame(imp).merge(FEAT_META.reset_index()[["feature", "group", "band_lo", "band_hi", "level_dep"]], on="feature") \
    .sort_values("perm_auc_drop", ascending=False).reset_index(drop=True)
savetab(PIMP.round(5), "T16_permutation_importance")
print(f"Permutation importance ({BEST_MODEL}, {'TEST' if HAS_TEST else 'DEV (không có TEST)'}; AUC khung gốc {base:.3f}):")
display(PIMP.head(15).round(4))

SHAP_OK = False
tree_models = [m for m in ("XGB", "RF") if m in cfg.models]
if HAS_SHAP and tree_models:
    _tm = E4_DEV[(E4_DEV.featset == "K_MAIN") & (E4_DEV.model.isin(tree_models))].sort_values("rec_auc_rep_mean").model.iloc[-1]
    try:
        _tf = FINAL[("K_MAIN", _tm)]; _Xs = _B[: min(len(_B), 1500)]
        sv = shap.TreeExplainer(_tf["model"]).shap_values(_Xs)
        sv = sv[1] if isinstance(sv, list) else (sv[..., 1] if np.ndim(sv) == 3 else sv)
        fig = plt.figure(figsize=(7, 6))
        shap.summary_plot(sv, _Xs, feature_names=FEAT_MAIN, max_display=20, show=False, plot_size=None)
        plt.title(f"SHAP summary - {_tm} ({'TEST' if HAS_TEST else 'DEV'} frames)")
        savefig(plt.gcf(), "Fig13_shap_summary", f"SHAP values of the {_tm} model.")
        SHAP_IMP = pd.Series(np.abs(sv).mean(0), index=FEAT_MAIN).sort_values(ascending=False)
        SHAP_OK = True
    except Exception as e:
        print("!! SHAP thất bại:", type(e).__name__, str(e)[:200])

fig, ax = plt.subplots(figsize=(9, 3.4))
_pm = PIMP.dropna(subset=["band_lo"])
for r in _pm.itertuples():
    ax.plot([r.band_lo, r.band_hi], [r.perm_auc_drop] * 2, color=GROUP_COLOR[r.group], lw=3, solid_capstyle="butt")
for g in sorted(set(_pm.group)):
    ax.plot([], [], color=GROUP_COLOR[g], lw=3, label=g)
ax.axhline(0, color=INK2, lw=0.8); freq_axis(ax, cfg.f_lo, CHAIN.f_hi); ax.set_ylabel("AUC drop when permuted")
ax.set_title(f"Frequency map of band-specific feature importance ({BEST_MODEL}, K_MAIN)"); ax.legend(fontsize=7)
fig.tight_layout()
savefig(fig, "Fig14_importance_frequency_map", "Permutation importance of band-specific features on the frequency axis.")

In [ ]:
# =========================================================
# 10.2 Confound mức tín hiệu, rò rỉ dữ liệu theo khung, thời gian nghe tối thiểu
# =========================================================
ROB = []
_lvl = run_cv(DS(BEST_COND, "dev", ["td_rms_db"]), [(None, 0, "RF"), (None, 0, "SVM")], tag="level-only")
for m in ("RF", "SVM"):
    s = summarize(_lvl, (None, 0, m)); ROB.append(dict(test="level only (RMS)", model=m, k=1, rec_auc=s["rec_auc_rep_mean"],
                                                       lo=s["rec_auc_lo"], hi=s["rec_auc_hi"]))
# chọn lại TRONG fold chỉ từ các đặc trưng KHÔNG phụ thuộc mức, cùng k = K_MAIN, cùng fold với E3 (ghép cặp)
_ginv = run_cv(DS(BEST_COND, "dev", [f for f in FEATURES if f not in LEVEL_DEP]),
               [("consensus", K_MAIN, m) for m in cfg.models], rankers=("consensus",), tag="gain-invariant")
for m in cfg.models:
    a, b = summarize(E3_RUN, ("consensus", K_MAIN, m)), summarize(_ginv, ("consensus", K_MAIN, m))
    c = compare(b, a, E3_RUN.ds.rec_y)
    ROB.append(dict(test="in-fold top-K_MAIN, all features", model=m, k=K_MAIN, rec_auc=a["rec_auc_rep_mean"],
                    lo=a["rec_auc_lo"], hi=a["rec_auc_hi"]))
    ROB.append(dict(test="in-fold top-K_MAIN, level-dependent features excluded", model=m, k=K_MAIN,
                    rec_auc=b["rec_auc_rep_mean"], lo=b["rec_auc_lo"], hi=b["rec_auc_hi"], d_vs_full=c["d_auc"],
                    p_delong=c["p_delong"]))
_full = run_cv(DS(BEST_COND, "dev", FEAT_MAIN), [(None, 0, BEST_MODEL)], tag="K_MAIN default (grouped)")

# rò rỉ dữ liệu: KFold ngẫu nhiên theo khung trên DEV (SAI) vs theo bản ghi (ĐÚNG, _full)
_d = _full.ds


def leak_job(tr, te, m, seed):
    prep = Prep().fit(_d.X[tr]); mdl = fit_model(m, prep.transform(_d.X[tr]), _d.y[tr], _d.g[tr], seed=seed)
    return te, score_model(m, mdl, prep.transform(_d.X[te]))


_kf = KFold(cfg.n_splits, shuffle=True, random_state=cfg.seeds[0]).split(_d.X)
_o = Parallel(n_jobs=N_JOBS, backend="loky")(delayed(leak_job)(tr, te, BEST_MODEL, 3) for tr, te in _kf)
_sr = np.full(len(_d.y), np.nan)
for te, sc in _o:
    _sr[te] = sc
_grp = summarize(_full, (None, 0, BEST_MODEL))
LEAKDEMO = pd.DataFrame([
    dict(protocol="A. grouped by recording (correct)", frame_auc=_grp["frame_auc"], rec_auc=_grp["rec_auc"]),
    dict(protocol="B. random frame split (leaky)", frame_auc=auc_safe(_d.y, _sr), rec_auc=auc_safe(_d.rec_y, rec_scores(_sr, _d.g, _d.recs)))])
LEAKDEMO.loc[2] = ["B − A (inflation)", LEAKDEMO.frame_auc[1] - LEAKDEMO.frame_auc[0], LEAKDEMO.rec_auc[1] - LEAKDEMO.rec_auc[0]]
ROB = pd.DataFrame(ROB)
display(Markdown("**Bảng 17. Kiểm tra độ bền (DEV CV, tham số mặc định).**")); display(ROB.round(4))
display(Markdown(f"**Bảng 18. Mức thổi phồng do chia fold theo khung ({BEST_MODEL}, K_MAIN).**")); display(LEAKDEMO.round(4))
savetab(ROB.round(4), "T17_robustness"); savetab(LEAKDEMO.round(4), "T18_leakage_demo")

# thời gian nghe tối thiểu: n khung LIÊN TỤC ngẫu nhiên trong mỗi bản ghi
if HAS_TEST:
    _lt_df, _lt_s, _lt_src = DST.df, TEST_SC[("K_MAIN", BEST_MODEL)][0], "TEST"
else:
    _lt_df, _lt_s, _lt_src = NESTED["K_MAIN"].ds.df, np.nanmean(NESTED["K_MAIN"].scores[(None, K_MAIN, BEST_MODEL)], 0), "DEV nested OOF"
_lt = _lt_df.assign(s=_lt_s).sort_values(["record_id", "file_stem", "frame_idx"])
_groups = {r: g.s.values for r, g in _lt.groupby("record_id")}
_ry = REC_LABEL.loc[list(_groups)].values
rng = np.random.default_rng(1); LT = []
for n in [1, 2, 3, 5, 8, 10, 15, 20, 30, 45]:
    aucs = []
    for _ in range(50 if cfg.smoke_test else 300):
        sc = []
        for v in _groups.values():
            if len(v) <= n:
                sc.append(v.mean())
            else:
                st_ = rng.integers(0, len(v) - n + 1); sc.append(v[st_: st_ + n].mean())
        aucs.append(auc_safe(_ry, sc))
    LT.append(dict(n_frames=n, seconds=n * cfg.hop_sec + (cfg.frame_sec - cfg.hop_sec), auc=np.mean(aucs),
                   lo=np.percentile(aucs, 2.5), hi=np.percentile(aucs, 97.5)))
LT = pd.DataFrame(LT)
AUC_ALL_FRAMES = auc_safe(_ry, [v.mean() for v in _groups.values()])     # nghe toàn bộ bản ghi
LT["auc_all_frames"] = AUC_ALL_FRAMES
LT_MIN_S = float(LT.seconds[LT.auc >= AUC_ALL_FRAMES - 0.02].min()) if (LT.auc >= AUC_ALL_FRAMES - 0.02).any() else np.nan
savetab(LT.round(4), "T19_listening_time")
print(f"AUC khi nghe toàn bộ bản ghi = {AUC_ALL_FRAMES:.3f}; nghe liên tục {LT_MIN_S:.0f} s đạt AUC >= toàn bộ - 0,02")
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.fill_between(LT.seconds, LT.lo, LT.hi, color=CAT[0], alpha=0.18, lw=0)
ax.plot(LT.seconds, LT.auc, "-o", color=CAT[0], ms=4, label="n contiguous frames")
ax.axhline(AUC_ALL_FRAMES, color=INK2, lw=0.8, label=f"whole recording ({AUC_ALL_FRAMES:.3f})"); ax.legend(loc="lower right")
ax.set_xscale("log"); ax.set_xlabel("continuous listening time per recording (s)"); ax.set_ylabel("record-level AUC")
ax.set_title(f"How long must the operator listen? ({BEST_MODEL}, {_lt_src})")
fig.tight_layout()
savefig(fig, "Fig15_listening_time", "Record-level AUC as a function of continuous listening time.")

In [ ]:
# =========================================================
# 10.3 Chi phí tính toán và phân tích lỗi theo bản ghi
# =========================================================
_r0 = files.iloc[0]; _x = load_raw(_r0.path, SR)
t0 = time.time(); _sig, _st, _q, _ = CHAIN.signals(_x); t_proc = (time.time() - t0) / (len(_x) / SR)
_n = min(20, len(_st)); t0 = time.time()
for s in _st[:_n]:
    FX(_sig[BEST_COND][s: s + CHAIN.F])
t_feat = (time.time() - t0) / max(_n, 1) * 1000
COST = [dict(item=f"signal chain C0-C6 (s CPU per s audio)", value=t_proc),
        dict(item=f"full feature extraction ({len(FEATURES)} features), ms / {cfg.frame_sec:g}-s frame", value=t_feat)]
_Xb = np.repeat(_B[:1], 1000, axis=0) if len(_B) else np.zeros((1000, len(FEAT_MAIN)), np.float32)
for m in cfg.models:
    t0 = time.time(); score_model(m, FINAL[("K_MAIN", m)]["model"], _Xb)
    COST.append(dict(item=f"{m} inference, ms / frame", value=(time.time() - t0)))
COST = pd.DataFrame(COST); display(COST.round(4)); savetab(COST.round(5), "T20_compute_cost")

if HAS_TEST:
    ERR = pd.DataFrame({m: TEST_SC[("K_MAIN", m)][1] for m in cfg.models}, index=DST.recs)
    ERR["label"] = DST.rec_y
    ERR["n_wrong"] = sum(((ERR[m] >= 0.5).astype(int) != ERR.label).astype(int) for m in cfg.models)
    ERR = ERR.join(records.set_index("record_id")[["duration_s", "n_files"]]) \
        .join(QC_FRAMES.groupby("record_id").keep_C3.mean().rsub(1).rename("rejected_frac")) \
        .join(FEAT["C0_RAW"].groupby("record_id").td_rms_db.median().rename("rms_db"))
    ERR_BAD = ERR[ERR.n_wrong >= max(1, len(cfg.models) // 2)].sort_values("n_wrong", ascending=False)
    display(Markdown(f"**Bảng 21. Bản ghi TEST bị ≥ {max(1, len(cfg.models) // 2)} mô hình phân loại sai.**"))
    display(ERR_BAD.round(3)); savetab(ERR.round(4).reset_index().rename(columns={"index": "record_id"}), "T21_test_error_analysis")

## 11. Tổng hợp kết quả cho bài báo / Results summary for the manuscript

Đoạn dưới đây được **sinh tự động từ số đo** của lần chạy này (không có số viết cứng). Sao chép vào mục
*Results* rồi biên tập văn phong. Toàn bộ bảng (`tables/*.csv`, `*.tex`) và hình (`figures/*.png`, `*.pdf`,
300 dpi) nằm trong thư mục đầu ra và được nén thành `leak_ml_outputs.zip`.

In [ ]:
# =========================================================
# 11.1 Đoạn kết quả tự sinh + tệp JSON các con số chính
# =========================================================
_m = E4_DEV[E4_DEV.featset == "K_MAIN"].set_index("model")
_bm = _m.loc[BEST_MODEL]
res_lines = [
    f"### Kết quả chính {'(DỮ LIỆU MÔ PHỎNG — KHÔNG DÙNG ĐỂ BÁO CÁO)' if SYNTHETIC else ''}",
    f"**Dữ liệu.** {len(records)} bản ghi độc lập ({int(records.label.sum())} leak / {int((1 - records.label).sum())} noleak), "
    f"{len(files)} tệp, {records.duration_s.sum() / 60:.1f} phút, {SR} Hz; khung {cfg.frame_sec:g} s. "
    f"DEV = {int((records.split == 'dev').sum())} bản ghi; TEST khoá = {int((records.split == 'test').sum())} bản ghi.",
    f"**Tiền xử lý.** Bộ lọc bất thường + không liên quan loại {100 * (1 - QC_FRAMES.keep_C3.mean()):.1f}% số khung "
    f"(leak {100 * rej.loc['rejected_C3', 'leak']:.1f}%, noleak {100 * rej.loc['rejected_C3', 'noleak']:.1f}%; "
    f"khác biệt giữa hai lớp: Mann–Whitney p = {p_rej:.3f}).",
    f"**RQ1 (dải tần).** Năng lượng tương đối cao hơn ở leak trong: {E1_SUMMARY['leak_higher']}; cao hơn ở noleak trong: "
    f"{E1_SUMMARY['noleak_higher']}. Cửa sổ một octave tốt nhất {E1_SUMMARY['best_octave']} (AUC {E1_SUMMARY['best_octave_auc']:.3f}; "
    f"toàn dải {E1_SUMMARY['auc_full']:.3f}). Quét dải: thông thấp đến {E1_SUMMARY['fc_lp']:.0f} Hz và thông cao từ "
    f"{E1_SUMMARY['fc_hp']:.0f} Hz giữ AUC trong phạm vi {cfg.e1_tol} so với toàn dải"
    + (f"; dải thiết yếu {E1_SUMMARY['ess']['f_lo']:.0f}–{E1_SUMMARY['ess']['f_hi']:.0f} Hz đạt AUC {E1_SUMMARY['ess']['auc']:.3f}."
       if E1_SUMMARY["ess"] else " (thông tin lặp lại ở cả hai phía phổ)."),
    f"**RQ2 (xử lý vs thô).** AUC mức bản ghi trung bình 4 mô hình: thô {E2_SUMMARY['raw_mean']:.3f} → "
    f"{E2_SUMMARY['best']} {E2_SUMMARY['best_mean']:.3f} (Δ {E2_SUMMARY['delta_mean']:+.3f}). Khác biệt có ý nghĩa sau Holm ở: "
    f"{', '.join(E2_SUMMARY['sig_better']) if E2_SUMMARY['sig_better'] else 'không mô hình nào'}."
    + (" Trên TEST: " + "; ".join(f"{r.model} {r.auc_raw:.3f}→{r.auc_best:.3f} (p_Holm {r.p_delong_holm:.3f})"
                                    for r in E4C.itertuples()) + "." if E4C is not None else ""),
    f"**RQ3 (chọn đặc trưng).** {len(FEATURES)} đặc trưng (~{N_CLUST90} cụm độc lập). Bộ tối thiểu **{K_MIN}** đặc trưng giữ "
    f"AUC trong phạm vi {cfg.k_tol} so với tốt nhất; bộ làm việc **{K_MAIN}** đặc trưng. Nhóm chiếm ưu thế trong K_MIN: "
    f"{', '.join(f'{g} ({n})' for g, n in SEL[SEL.in_K_MIN].group.value_counts().items())}.",
    f"**Mô hình (CV lồng, DEV, K_MAIN).** " + "; ".join(
        f"{m}: AUC {_m.loc[m, 'rec_auc_rep_mean']:.3f} ± {_m.loc[m, 'rec_auc_rep_sd']:.3f}, BAcc {_m.loc[m, 'rec_bacc']:.3f}, "
        f"F1 {_m.loc[m, 'rec_f1']:.3f}" for m in cfg.models) + f". Tốt nhất: **{BEST_MODEL}**.",
]
if HAS_TEST:
    _t = E4_TEST[E4_TEST.featset == "K_MAIN"].set_index("model")
    res_lines.append(f"**TEST khoá ({len(DST.recs)} bản ghi, K_MAIN).** " + "; ".join(
        f"{m}: AUC {fmt_ci(_t.loc[m, 'rec_auc'], _t.loc[m, 'rec_auc_lo'], _t.loc[m, 'rec_auc_hi'])}, "
        f"Sens {_t.loc[m, 'rec_sens']:.2f}, Spec {_t.loc[m, 'rec_spec']:.2f}" for m in cfg.models) + ".")
res_lines.append(
    f"**Độ bền.** Chỉ mức RMS: AUC {ROB[ROB.test.str.startswith('level only')].rec_auc.max():.3f}; chọn lại không dùng đặc trưng "
    f"phụ thuộc mức ({BEST_MODEL}): ΔAUC {ROB[(ROB.test.str.contains('excluded')) & (ROB.model == BEST_MODEL)].d_vs_full.iloc[0]:+.3f}. "
    f"Chia fold theo khung làm AUC bản ghi tăng giả {LEAKDEMO.rec_auc[2]:+.3f}. "
    f"Nghe liên tục {LT_MIN_S:.0f} s đạt AUC trong phạm vi 0,02 so với nghe toàn bộ bản ghi (AUC {AUC_ALL_FRAMES:.3f}).")
RESULTS_MD = "\n\n".join(res_lines)
display(Markdown(RESULTS_MD))
(OUT / "RESULTS_summary.md").write_text(RESULTS_MD, encoding="utf-8")

KEY = dict(synthetic=bool(SYNTHETIC), n_records=int(len(records)), n_files=int(len(files)), sr=SR, frame_sec=cfg.frame_sec,
           conditions=list(cfg.conditions), best_condition=BEST_COND, e1=E1_SUMMARY, e2=E2_SUMMARY, K_MIN=K_MIN, K_MAIN=K_MAIN,
           features_K_MIN=FEAT_MIN, features_K_MAIN=FEAT_MAIN, best_model=BEST_MODEL,
           dev_nested={m: dict(auc=float(_m.loc[m, "rec_auc_rep_mean"]), sd=float(_m.loc[m, "rec_auc_rep_sd"])) for m in cfg.models},
           test=({m: float(E4_TEST[(E4_TEST.featset == 'K_MAIN') & (E4_TEST.model == m)].rec_auc.iloc[0]) for m in cfg.models}
                 if HAS_TEST else None),
           config=json.loads(json.dumps(asdict(cfg), default=str)))
(OUT / "key_results.json").write_text(json.dumps(KEY, indent=1, ensure_ascii=False, default=str), encoding="utf-8")

In [ ]:
# =========================================================
# 11.2 Danh mục hình/bảng + nén đầu ra
# =========================================================
display(Markdown("**Danh mục hình**\n\n" + "\n".join(f"- `{n}` — {c}" for n, c in FIG_LOG)))
display(Markdown("**Danh mục bảng**\n\n" + "\n".join(f"- `{p.name}`" for p in sorted(TAB_DIR.glob("*.csv")))))
_zip = shutil.make_archive(str(OUT.parent / "leak_ml_outputs"), "zip", root_dir=OUT,
                           base_dir=".") if not cfg.smoke_test else None
print("Đã nén:", _zip if _zip else "(bỏ qua ở smoke test)")
print("Mô hình đã lưu:", sorted(p.name for p in MODEL_DIR.glob("*.joblib")))

### Dùng mô hình đã lưu cho bản ghi mới / Inference on a new recording

```python
import joblib, numpy as np
b = joblib.load("/kaggle/working/leak_ml/models/final_SVM_K_MAIN_k30.joblib")   # tên tệp thực tế ở mục 11.2
# 1) chạy SignalChain(b["sr"], PARAMS).signals(x) và lấy tín hiệu + mặt nạ khung của b["condition"]
# 2) trích đặc trưng từng khung bằng FeatureExtractor, giữ đúng thứ tự cột b["features"]
# 3) chuẩn hoá: z = clip((X - b["prep"]["mu"]) / b["prep"]["sd"], -8, 8) sau khi điền NaN bằng b["prep"]["med"]
# 4) điểm bản ghi = trung bình xác suất các khung; > 0,5 -> nghi rò rỉ
```

**Lưu ý khi viết bài**: nếu thư mục đầu ra có dòng *SYNTHETIC DEMO MODE*, mọi con số chỉ để kiểm tra
chương trình. Hãy gắn dataset thật và chạy lại toàn bộ trước khi trích dẫn.